# Handwritten Circuit Detector
Junyi Liu — APS360 individual course project.
This is an archived experimental notebook, not a verified run-all pipeline. See README.md for data requirements, section order, results provenance and citations. Outputs were cleared for repository distribution; code is unchanged.

In [ ]:
import random
import shutil
import csv
import numpy as np
import torch
import json
import zipfile
from collections import Counter
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch.utils.data import Dataset, DataLoader,ConcatDataset,WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
from skimage.feature import hog
from sklearn.svm import LinearSVC
import torch.nn as nn
import torch.nn.functional as Func
import torchvision.transforms.functional as F
import torchvision.models as models
from collections import defaultdict
from torchvision.ops import box_iou, nms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report,precision_recall_fscore_support,ConfusionMatrixDisplay, confusion_matrix

Connect the database to colab

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Data splitting

In [ ]:
!cp -r "/content/gdrive/MyDrive/APS360/Project/merged_data" "/content/merged_data"

In [ ]:
!cp -r "/content/gdrive/MyDrive/APS360/Project/synthetic_train_only" "/content/synthetic_train"

In [ ]:
data_root = Path("/content/gdrive/MyDrive/APS360/Project/Data")
output_root = Path("/content/merged_data")
synthetic_root = Path("/content/synthetic_train")

cnn_root = Path("/content/cnn_cropped_symbols")
yolo_root = Path("/content/yolo_dataset")

In [ ]:


random.seed(1000)
xml_files = sorted(list(output_root.rglob("*.xml")))
print("Number of real XML files:", len(xml_files))

# # create folders
# for folder in ["images", "annotations", "segmentation", "instances"]:
#     for split in ["train", "val", "test"]:
#         (output_root / folder / split).mkdir(parents=True, exist_ok=True)

# samples = []

# # collect samples
# for drafter_dir in sorted(data_root.glob("drafter_*")):
#     drafter_id = int(drafter_dir.name.split("_")[1])

#     # optional: skip drafter_0 and negative drafters
#     if drafter_id <= 0:
#         continue

#     img_dir = drafter_dir / "images"
#     ann_dir = drafter_dir / "annotations"
#     seg_dir = drafter_dir / "segmentation"
#     ins_dir = drafter_dir / "instances"

#     if not img_dir.exists() or not ann_dir.exists():
#         continue

#     for img_path in img_dir.iterdir():
#         if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
#             continue

#         xml_path = ann_dir / f"{img_path.stem}.xml"

#         if not xml_path.exists():
#             continue

#         seg_path = None
#         for ext in [".jpg", ".jpeg", ".png"]:
#             candidate = seg_dir / f"{img_path.stem}{ext}"
#             if candidate.exists():
#                 seg_path = candidate
#                 break

#         ins_path = ins_dir / f"{img_path.stem}.json"
#         if not ins_path.exists():
#             ins_path = None

#         sample_id = f"{drafter_dir.name}_{img_path.stem}"

#         samples.append({
#             "id": sample_id,
#             "image": img_path,
#             "annotation": xml_path,
#             "segmentation": seg_path,
#             "instance": ins_path
#         })

# print("Total samples:", len(samples))

# # split 80/10/10
# random.shuffle(samples)

# n = len(samples)
# train_end = int(0.8 * n)
# val_end = int(0.9 * n)

# splits = {
#     "train": samples[:train_end],
#     "val": samples[train_end:val_end],
#     "test": samples[val_end:]
# }

# # copy files
# metadata_rows = []

# for split, split_samples in splits.items():
#     for s in split_samples:
#         new_img_name = s["id"] + s["image"].suffix.lower()
#         new_ann_name = s["id"] + ".xml"

#         shutil.copy2(s["image"], output_root / "images" / split / new_img_name)
#         shutil.copy2(s["annotation"], output_root / "annotations" / split / new_ann_name)

#         new_seg_name = ""
#         new_ins_name = ""

#         if s["segmentation"] is not None:
#             new_seg_name = s["id"] + s["segmentation"].suffix.lower()
#             shutil.copy2(s["segmentation"], output_root / "segmentation" / split / new_seg_name)

#         if s["instance"] is not None:
#             new_ins_name = s["id"] + ".json"
#             shutil.copy2(s["instance"], output_root / "instances" / split / new_ins_name)

#         metadata_rows.append([
#             s["id"],
#             split,
#             new_img_name,
#             new_ann_name,
#             new_seg_name,
#             new_ins_name
#         ])

# # save metadata
# metadata_path = output_root / "metadata.csv"

# with open(metadata_path, "w", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerow(["sample_id", "split", "image", "annotation", "segmentation", "instance"])
#     writer.writerows(metadata_rows)

# print("Done.")
# print("Train:", len(splits["train"]))
# print("Val:", len(splits["val"]))
# print("Test:", len(splits["test"]))
# print("Metadata saved to:", metadata_path)

In [ ]:


def image_names(split):
    return set(p.name for p in (output_root / "images" / split).glob("*"))

train = image_names("train")
val = image_names("val")
test = image_names("test")

print("train ∩ val:", len(train & val))
print("train ∩ test:", len(train & test))
print("val ∩ test:", len(val & test))

for split in ["train", "val", "test"]:
    imgs = set(p.stem for p in (output_root / "images" / split).glob("*"))
    anns = set(p.stem for p in (output_root / "annotations" / split).glob("*.xml"))

    print("\n", split)
    print("images:", len(imgs))
    print("annotations:", len(anns))
    print("missing annotations:", len(imgs - anns))
    print("extra annotations:", len(anns - imgs))

In [ ]:

def image_names(split):
    return set(p.name for p in (synthetic_root / "images" / split).glob("*"))

train = image_names("train")
val = image_names("val")
test = image_names("test")

print("train ∩ val:", len(train & val))
print("train ∩ test:", len(train & test))
print("val ∩ test:", len(val & test))

for split in ["train", "val", "test"]:
    imgs = set(p.stem for p in (synthetic_root / "images" / split).glob("*"))
    anns = set(p.stem for p in (synthetic_root / "annotations" / split).glob("*.xml"))

    print("\n", split)
    print("images:", len(imgs))
    print("annotations:", len(anns))
    print("missing annotations:", len(imgs - anns))
    print("extra annotations:", len(anns - imgs))

In [ ]:
imgs = list((output_root / "images" / "train").glob("*"))

In [ ]:
split = "train"
img_path = random.choice(list((output_root / "images" / split).glob("*")))
xml_path = output_root / "annotations" / split / (img_path.stem + ".xml")

img = Image.open(img_path).convert("RGB")

tree = ET.parse(xml_path)
xml_root = tree.getroot()

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(img)

for obj in xml_root.findall("object"):
    cls = obj.find("name").text
    box = obj.find("bndbox")

    xmin = int(box.find("xmin").text)
    ymin = int(box.find("ymin").text)
    xmax = int(box.find("xmax").text)
    ymax = int(box.find("ymax").text)

    rect = patches.Rectangle(
        (xmin, ymin),
        xmax - xmin,
        ymax - ymin,
        fill=False,
        linewidth=2
    )

    ax.add_patch(rect)
    ax.text(xmin, ymin, cls, fontsize=8)

plt.axis("off")
plt.show()

print(img_path.name)

In [ ]:
classes_json = "/content/gdrive/MyDrive/APS360/Project/Data/classes.json"

transform = transforms.Compose([
    transforms.Resize((448,448)),
    transforms.ToTensor()
])

def collate_fn(batch):
    return tuple(zip(*batch))


In [ ]:
class CircuitDetectionDataset(Dataset):
    def __init__(self, root, split="train", classes_json=None, img_size=448):
        self.root = Path(root)
        self.split = split
        self.image_dir = self.root / "images" / split
        self.ann_dir = self.root / "annotations" / split
        self.img_size = img_size

        with open(classes_json, "r") as f:
            self.class_to_id = json.load(f)

        self.images = sorted([
            p for p in self.image_dir.glob("*")
            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ])

        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        xml_path = self.ann_dir / (img_path.stem + ".xml")

        image = Image.open(img_path).convert("RGB")
        orig_w, orig_h = image.size

        boxes = []
        labels = []

        tree = ET.parse(xml_path)
        xml_root = tree.getroot()

        for obj in xml_root.findall("object"):
            cls_name = obj.find("name").text
            if cls_name not in self.class_to_id:
                continue

            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            # scale boxes to resized image
            xmin = xmin * self.img_size / orig_w
            xmax = xmax * self.img_size / orig_w
            ymin = ymin * self.img_size / orig_h
            ymax = ymax * self.img_size / orig_h

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_id[cls_name])

        image = self.transform(image)

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.long),
            "image_id": img_path.stem
        }

        return image, target

Baseline model

In [ ]:
!pip install -q scikit-image scikit-learn joblib

In [ ]:


CGHD_ROOT = Path(
    "/content/gdrive/MyDrive/APS360/Project/merged_data"
)

SYNTHETIC_ROOT = Path("/content/synthetic_train")


def show_structure(root, max_depth=3):
    print("=" * 70)
    print("Root:", root)
    print("Exists:", root.exists())

    if not root.exists():
        return

    for current_dir, subdirs, files in os.walk(root):
        current_path = Path(current_dir)
        depth = len(current_path.relative_to(root).parts)

        if depth > max_depth:
            subdirs[:] = []
            continue

        indent = "    " * depth
        print(f"{indent}{current_path.name}/")

        # Show only a few files from each directory
        for filename in sorted(files)[:5]:
            print(f"{indent}    {filename}")

        if len(files) > 5:
            print(f"{indent}    ... {len(files) - 5} more files")


def count_file_types(root):
    counts = Counter(
        path.suffix.lower()
        for path in root.rglob("*")
        if path.is_file()
    )

    print("\nFile counts:")
    for extension, count in counts.most_common():
        print(f"  {extension or '[no extension]'}: {count}")


show_structure(CGHD_ROOT)
count_file_types(CGHD_ROOT)

show_structure(SYNTHETIC_ROOT)
count_file_types(SYNTHETIC_ROOT)

In [ ]:


training_sources = [
    {
        "name": "CGHD",
        "image_dir": CGHD_ROOT / "images" / "train",
        "annotation_dir": CGHD_ROOT / "annotations" / "train"
    },
    {
        "name": "Synthetic",
        "image_dir": SYNTHETIC_ROOT / "images" / "train",
        "annotation_dir": SYNTHETIC_ROOT / "annotations" / "train"
    }
]

val_source = {
    "name": "CGHD validation",
    "image_dir": CGHD_ROOT / "images" / "val",
    "annotation_dir": CGHD_ROOT / "annotations" / "val"
}

test_source = {
    "name": "CGHD test",
    "image_dir": CGHD_ROOT / "images" / "test",
    "annotation_dir": CGHD_ROOT / "annotations" / "test"
}

In [ ]:
def read_annotation_classes(annotation_dir):
    class_counts = Counter()
    invalid_objects = 0
    total_objects = 0

    xml_paths = sorted(annotation_dir.glob("*.xml"))

    for xml_path in xml_paths:
        try:
            root = ET.parse(xml_path).getroot()

            for obj in root.findall("object"):
                class_name = obj.findtext("name")

                if class_name is None:
                    invalid_objects += 1
                    continue

                class_name = class_name.strip()
                bbox = obj.find("bndbox")

                if not class_name or bbox is None:
                    invalid_objects += 1
                    continue

                class_counts[class_name] += 1
                total_objects += 1

        except Exception as error:
            print("Failed to read:", xml_path.name, error)

    return {
        "xml_count": len(xml_paths),
        "total_objects": total_objects,
        "invalid_objects": invalid_objects,
        "class_counts": class_counts
    }

In [ ]:
scan_results = {}

for source in training_sources + [val_source, test_source]:
    result = read_annotation_classes(source["annotation_dir"])
    scan_results[source["name"]] = result

    print("=" * 60)
    print(source["name"])
    print("XML files:", result["xml_count"])
    print("Annotated objects:", result["total_objects"])
    print("Unique classes:", len(result["class_counts"]))
    print("Invalid objects:", result["invalid_objects"])
    print("Classes:")
    print(sorted(result["class_counts"].keys()))

In [ ]:
cghd_classes = set(
    scan_results["CGHD"]["class_counts"].keys()
)

synthetic_classes = set(
    scan_results["Synthetic"]["class_counts"].keys()
)

common_classes = cghd_classes & synthetic_classes
only_cghd = cghd_classes - synthetic_classes
only_synthetic = synthetic_classes - cghd_classes

print("CGHD classes:", len(cghd_classes))
print("Synthetic classes:", len(synthetic_classes))
print("Common classes:", len(common_classes))

print("\nOnly in CGHD:")
print(sorted(only_cghd))

print("\nOnly in synthetic:")
print(sorted(only_synthetic))

In [ ]:
for source_name in ["CGHD", "Synthetic"]:
    print(f"\nTop classes in {source_name}:")

    for class_name, count in scan_results[
        source_name
    ]["class_counts"].most_common(15):
        print(f"{class_name:25s} {count}")

In [ ]:
from pathlib import Path
from collections import defaultdict
import xml.etree.ElementTree as ET
import pandas as pd
import random

random.seed(1000)

CGHD_ROOT = Path(
    "/content/gdrive/MyDrive/APS360/Project/merged_data"
)

SYNTHETIC_ROOT = Path("/content/synthetic_train")

HOG_ROOT = Path("/content/hog_baseline")
HOG_ROOT.mkdir(parents=True, exist_ok=True)

training_sources = [
    {
        "name": "CGHD",
        "image_dir": CGHD_ROOT / "images" / "train",
        "annotation_dir": CGHD_ROOT / "annotations" / "train"
    },
    {
        "name": "Synthetic",
        "image_dir": SYNTHETIC_ROOT / "images" / "train",
        "annotation_dir": SYNTHETIC_ROOT / "annotations" / "train"
    }
]

IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg"]
MAX_POSITIVES_PER_CLASS = 500


In [ ]:
import json
from pathlib import Path

CLASSES_JSON = Path(
    "/content/gdrive/MyDrive/APS360/Project/Data/classes.json"
)

with open(CLASSES_JSON, "r") as file:
    class_data = json.load(file)

print(type(class_data))
print(class_data)

In [ ]:
if isinstance(class_data, list):
    # Example: ["resistor", "capacitor", ...]
    primary_class_names = class_data

elif isinstance(class_data, dict):
    # Possible format: {"resistor": 0, "capacitor": 1, ...}
    if all(isinstance(value, int) for value in class_data.values()):
        primary_class_names = [
            class_name
            for class_name, class_id in sorted(
                class_data.items(),
                key=lambda item: item[1]
            )
        ]

    # Possible format: {"0": "resistor", "1": "capacitor", ...}
    elif all(str(key).isdigit() for key in class_data.keys()):
        primary_class_names = [
            class_name
            for class_id, class_name in sorted(
                class_data.items(),
                key=lambda item: int(item[0])
            )
        ]

    # Possible format: {"classes": [...]}
    elif "classes" in class_data:
        primary_class_names = class_data["classes"]

    else:
        raise ValueError(
            "Unrecognized classes.json dictionary format."
        )

else:
    raise ValueError(
        "classes.json must contain a list or dictionary."
    )

class_to_id = {
    class_name: class_id
    for class_id, class_name in enumerate(primary_class_names)
}

id_to_class = {
    class_id: class_name
    for class_name, class_id in class_to_id.items()
}

print("Number of classes:", len(primary_class_names))
print("First 10 classes:")

for class_id, class_name in list(id_to_class.items())[:10]:
    print(class_id, class_name)

In [ ]:


primary_class_names = list(primary_class_names)

class_to_id = {
    class_name: class_id
    for class_id, class_name in enumerate(primary_class_names)
}

print("Primary-model classes:", len(primary_class_names))

In [ ]:
def find_image_path(image_dir, xml_path, xml_root):
    filename = xml_root.findtext("filename")

    if filename:
        candidate = image_dir / Path(filename).name

        if candidate.exists():
            return candidate

    for extension in IMAGE_EXTENSIONS:
        candidate = image_dir / f"{xml_path.stem}{extension}"

        if candidate.exists():
            return candidate

    return None

In [ ]:
def crop_with_padding(image, xmin, ymin, xmax, ymax, padding_ratio=0.15):
    width, height = image.size

    box_width = xmax - xmin
    box_height = ymax - ymin

    pad_x = int(box_width * padding_ratio)
    pad_y = int(box_height * padding_ratio)

    xmin = max(0, xmin - pad_x)
    ymin = max(0, ymin - pad_y)
    xmax = min(width, xmax + pad_x)
    ymax = min(height, ymax + pad_y)

    return image.crop((xmin, ymin, xmax, ymax))

In [ ]:
samples_by_class_source = defaultdict(
    lambda: defaultdict(list)
)

invalid_boxes = 0
missing_images = 0
unknown_classes = set()

for source in training_sources:
    source_name = source["name"]
    image_dir = source["image_dir"]
    annotation_dir = source["annotation_dir"]

    for xml_path in sorted(annotation_dir.glob("*.xml")):
        try:
            xml_root = ET.parse(xml_path).getroot()

            image_path = find_image_path(
                image_dir,
                xml_path,
                xml_root
            )

            if image_path is None:
                missing_images += 1
                continue

            for obj in xml_root.findall("object"):
                class_name = obj.findtext("name")

                if class_name is None:
                    continue

                class_name = class_name.strip()

                if class_name not in class_to_id:
                    unknown_classes.add(class_name)
                    continue

                bbox = obj.find("bndbox")

                if bbox is None:
                    invalid_boxes += 1
                    continue

                xmin = int(float(bbox.findtext("xmin")))
                ymin = int(float(bbox.findtext("ymin")))
                xmax = int(float(bbox.findtext("xmax")))
                ymax = int(float(bbox.findtext("ymax")))

                if xmax <= xmin or ymax <= ymin:
                    invalid_boxes += 1
                    continue

                samples_by_class_source[class_name][
                    source_name
                ].append({
                    "source": source_name,
                    "image_path": str(image_path),
                    "xml_path": str(xml_path),
                    "class_name": class_name,
                    "class_id": class_to_id[class_name],
                    "xmin": xmin,
                    "ymin": ymin,
                    "xmax": xmax,
                    "ymax": ymax
                })

        except Exception as error:
            print("Failed to read:", xml_path, error)

print("Missing images:", missing_images)
print("Invalid boxes:", invalid_boxes)
print("Classes not found in primary mapping:", sorted(unknown_classes))

In [ ]:
selected_samples = []

for class_name in primary_class_names:
    source_samples = samples_by_class_source[class_name]

    cghd_samples = source_samples.get("CGHD", []).copy()
    synthetic_samples = source_samples.get("Synthetic", []).copy()

    random.shuffle(cghd_samples)
    random.shuffle(synthetic_samples)

    class_selection = []

    if synthetic_samples:
        # Try to use half real and half synthetic.
        target_real = MAX_POSITIVES_PER_CLASS // 2
        target_synthetic = (
            MAX_POSITIVES_PER_CLASS - target_real
        )

        selected_real = cghd_samples[:target_real]
        selected_synthetic = synthetic_samples[
            :target_synthetic
        ]

        class_selection.extend(selected_real)
        class_selection.extend(selected_synthetic)

        # Fill remaining spaces from either source.
        remaining = (
            MAX_POSITIVES_PER_CLASS
            - len(class_selection)
        )

        unused_real = cghd_samples[len(selected_real):]
        unused_synthetic = synthetic_samples[
            len(selected_synthetic):
        ]

        additional_samples = (
            unused_real + unused_synthetic
        )

        random.shuffle(additional_samples)

        class_selection.extend(
            additional_samples[:remaining]
        )

    else:
        class_selection = cghd_samples[
            :MAX_POSITIVES_PER_CLASS
        ]

    selected_samples.extend(class_selection)

In [ ]:
positive_manifest = pd.DataFrame(selected_samples)

manifest_path = HOG_ROOT / "positive_manifest.csv"
positive_manifest.to_csv(manifest_path, index=False)

print("Saved:", manifest_path)
print("Selected positive samples:", len(positive_manifest))
print("Classes represented:", positive_manifest["class_name"].nunique())

In [ ]:
distribution = (
    positive_manifest
    .groupby(["class_name", "source"])
    .size()
    .unstack(fill_value=0)
)

distribution["total"] = distribution.sum(axis=1)

display(distribution.sort_values("total", ascending=False))

In [ ]:

sample_rows = positive_manifest.sample(
    min(12, len(positive_manifest)),
    random_state=1000
)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))

for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    image = Image.open(row["image_path"]).convert("RGB")

    crop = crop_with_padding(
        image,
        int(row["xmin"]),
        int(row["ymin"]),
        int(row["xmax"]),
        int(row["ymax"]),
        padding_ratio=0.15
    )

    ax.imshow(crop)
    ax.set_title(
        f'{row["class_name"]}\n{row["source"]}',
        fontsize=8
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
HOG_ROOT = Path("/content/hog_baseline")

positive_manifest = pd.read_csv(
    HOG_ROOT / "positive_manifest.csv"
)

print("Positive samples:", len(positive_manifest))
print("Classes:", positive_manifest["class_name"].nunique())

In [ ]:
HOG_IMAGE_SIZE = 64

def crop_with_padding(
    image,
    xmin,
    ymin,
    xmax,
    ymax,
    padding_ratio=0.15
):
    image_width, image_height = image.size

    box_width = xmax - xmin
    box_height = ymax - ymin

    pad_x = int(box_width * padding_ratio)
    pad_y = int(box_height * padding_ratio)

    xmin = max(0, xmin - pad_x)
    ymin = max(0, ymin - pad_y)
    xmax = min(image_width, xmax + pad_x)
    ymax = min(image_height, ymax + pad_y)

    return image.crop((xmin, ymin, xmax, ymax))

In [ ]:
def extract_hog_feature(image):
    image = image.convert("L")
    image = image.resize(
        (HOG_IMAGE_SIZE, HOG_IMAGE_SIZE)
    )

    image_array = np.asarray(
        image,
        dtype=np.float32
    ) / 255.0

    feature = hog(
        image_array,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    )

    return feature.astype(np.float32)

In [ ]:
!pip install -q scikit-image tqdm

from tqdm.auto import tqdm

In [ ]:
positive_features = []
positive_labels = []
valid_rows = []
failed_samples = []
!pip install -q scikit-image tqdm
for row_index, row in tqdm(
    positive_manifest.iterrows(),
    total=len(positive_manifest),
    desc="Extracting positive HOG features"
):
    try:
        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        crop = crop_with_padding(
            image,
            int(row["xmin"]),
            int(row["ymin"]),
            int(row["xmax"]),
            int(row["ymax"]),
            padding_ratio=0.15
        )

        if crop.width < 4 or crop.height < 4:
            failed_samples.append({
                "row_index": row_index,
                "reason": "Crop too small"
            })
            continue

        feature = extract_hog_feature(crop)

        positive_features.append(feature)
        positive_labels.append(int(row["class_id"]))
        valid_rows.append(row_index)

    except Exception as error:
        failed_samples.append({
            "row_index": row_index,
            "reason": str(error)
        })

In [ ]:
X_positive = np.asarray(
    positive_features,
    dtype=np.float32
)

y_positive = np.asarray(
    positive_labels,
    dtype=np.int64
)

print("Feature matrix:", X_positive.shape)
print("Label array:", y_positive.shape)
print("Failed samples:", len(failed_samples))

In [ ]:
np.savez_compressed(
    HOG_ROOT / "positive_hog_features.npz",
    X=X_positive,
    y=y_positive
)

valid_positive_manifest = positive_manifest.loc[
    valid_rows
].reset_index(drop=True)

valid_positive_manifest.to_csv(
    HOG_ROOT / "valid_positive_manifest.csv",
    index=False
)

print("Saved positive HOG features.")

In [ ]:
positive_distribution = (
    valid_positive_manifest
    .groupby(["class_name", "source"])
    .size()
    .unstack(fill_value=0)
)

positive_distribution["total"] = (
    positive_distribution.sum(axis=1)
)

display(
    positive_distribution.sort_values(
        "total",
        ascending=False
    )
)

In [ ]:
def read_boxes_from_xml(xml_path):
    root = ET.parse(xml_path).getroot()
    boxes = []

    for obj in root.findall("object"):
        bbox = obj.find("bndbox")

        if bbox is None:
            continue

        xmin = int(float(bbox.findtext("xmin")))
        ymin = int(float(bbox.findtext("ymin")))
        xmax = int(float(bbox.findtext("xmax")))
        ymax = int(float(bbox.findtext("ymax")))

        if xmax > xmin and ymax > ymin:
            boxes.append([xmin, ymin, xmax, ymax])

    return np.asarray(boxes, dtype=np.float32).reshape(-1, 4)

In [ ]:
def maximum_iou(candidate_box, ground_truth_boxes):
    if len(ground_truth_boxes) == 0:
        return 0.0

    candidate_box = np.asarray(
        candidate_box,
        dtype=np.float32
    )

    x1 = np.maximum(candidate_box[0], ground_truth_boxes[:, 0])
    y1 = np.maximum(candidate_box[1], ground_truth_boxes[:, 1])
    x2 = np.minimum(candidate_box[2], ground_truth_boxes[:, 2])
    y2 = np.minimum(candidate_box[3], ground_truth_boxes[:, 3])

    intersection_width = np.maximum(0, x2 - x1)
    intersection_height = np.maximum(0, y2 - y1)
    intersection = intersection_width * intersection_height

    candidate_area = (
        (candidate_box[2] - candidate_box[0])
        * (candidate_box[3] - candidate_box[1])
    )

    ground_truth_areas = (
        (ground_truth_boxes[:, 2] - ground_truth_boxes[:, 0])
        * (ground_truth_boxes[:, 3] - ground_truth_boxes[:, 1])
    )

    union = candidate_area + ground_truth_areas - intersection

    ious = intersection / np.maximum(union, 1e-8)

    return float(np.max(ious))

In [ ]:
positive_box_widths = (
    positive_manifest["xmax"]
    - positive_manifest["xmin"]
).astype(int).to_numpy()

positive_box_heights = (
    positive_manifest["ymax"]
    - positive_manifest["ymin"]
).astype(int).to_numpy()

valid_sizes = (
    (positive_box_widths >= 8)
    & (positive_box_heights >= 8)
)

positive_box_widths = positive_box_widths[valid_sizes]
positive_box_heights = positive_box_heights[valid_sizes]

print("Available box sizes:", len(positive_box_widths))
print(
    "Median box size:",
    int(np.median(positive_box_widths)),
    "x",
    int(np.median(positive_box_heights))
)

In [ ]:
BACKGROUND_TARGET_PER_SOURCE = 6000
MAX_BACKGROUNDS_PER_IMAGE = 4
MAX_ATTEMPTS_PER_IMAGE = 30
BACKGROUND_IOU_THRESHOLD = 0.05

background_features = []
background_records = []
failed_background_images = []

In [ ]:

for source in training_sources:
    source_name = source["name"]
    image_dir = source["image_dir"]
    annotation_dir = source["annotation_dir"]

    xml_paths = list(annotation_dir.glob("*.xml"))
    random.shuffle(xml_paths)

    source_background_count = 0

    progress = tqdm(
        total=BACKGROUND_TARGET_PER_SOURCE,
        desc=f"Background samples: {source_name}"
    )

    try:
        for xml_path in xml_paths:
            if source_background_count >= BACKGROUND_TARGET_PER_SOURCE:
                break

            try:
                xml_root = ET.parse(xml_path).getroot()

                image_path = find_image_path(
                    image_dir,
                    xml_path,
                    xml_root
                )

                if image_path is None:
                    failed_background_images.append({
                        "xml_path": str(xml_path),
                        "reason": "Matching image not found"
                    })
                    continue

                image = Image.open(image_path).convert("RGB")
                image_width, image_height = image.size

                ground_truth_boxes = read_boxes_from_xml(xml_path)

                samples_from_image = 0
                attempts = 0

                while (
                    samples_from_image < MAX_BACKGROUNDS_PER_IMAGE
                    and attempts < MAX_ATTEMPTS_PER_IMAGE
                    and source_background_count < BACKGROUND_TARGET_PER_SOURCE
                ):
                    attempts += 1

                    size_index = random.randrange(
                        len(positive_box_widths)
                    )

                    box_width = int(
                        positive_box_widths[size_index]
                    )
                    box_height = int(
                        positive_box_heights[size_index]
                    )

                    # Limit extremely small or large windows.
                    box_width = max(
                        16,
                        min(box_width, image_width - 1, 192)
                    )
                    box_height = max(
                        16,
                        min(box_height, image_height - 1, 192)
                    )

                    if box_width <= 0 or box_height <= 0:
                        continue

                    if (
                        box_width >= image_width
                        or box_height >= image_height
                    ):
                        continue

                    xmin = random.randint(
                        0,
                        image_width - box_width
                    )
                    ymin = random.randint(
                        0,
                        image_height - box_height
                    )

                    xmax = xmin + box_width
                    ymax = ymin + box_height

                    candidate_box = [
                        xmin,
                        ymin,
                        xmax,
                        ymax
                    ]

                    overlap = maximum_iou(
                        candidate_box,
                        ground_truth_boxes
                    )

                    if overlap >= BACKGROUND_IOU_THRESHOLD:
                        continue

                    patch = image.crop(
                        (xmin, ymin, xmax, ymax)
                    )

                    feature = extract_hog_feature(patch)

                    background_features.append(feature)

                    background_records.append({
                        "source": source_name,
                        "image_path": str(image_path),
                        "xml_path": str(xml_path),
                        "xmin": xmin,
                        "ymin": ymin,
                        "xmax": xmax,
                        "ymax": ymax,
                        "max_iou": overlap
                    })

                    source_background_count += 1
                    samples_from_image += 1
                    progress.update(1)

            except Exception as error:
                failed_background_images.append({
                    "xml_path": str(xml_path),
                    "reason": str(error)
                })

    finally:
        progress.close()

    print(
        f"{source_name} background samples: "
        f"{source_background_count}"
    )

print("\nTotal background samples:", len(background_features))
print("Failed images:", len(failed_background_images))

In [ ]:
BACKGROUND_ID = len(primary_class_names)

X_background = np.asarray(
    background_features,
    dtype=np.float32
)

y_background = np.full(
    len(X_background),
    BACKGROUND_ID,
    dtype=np.int64
)

print("Background ID:", BACKGROUND_ID)
print("Background feature matrix:", X_background.shape)
print("Background labels:", y_background.shape)
print("Failed images:", len(failed_background_images))

In [ ]:
print("Contains NaN:", np.isnan(X_background).any())
print("Contains infinity:", np.isinf(X_background).any())

In [ ]:
background_manifest = pd.DataFrame(
    background_records
)

background_manifest.to_csv(
    HOG_ROOT / "background_manifest.csv",
    index=False
)

np.savez_compressed(
    HOG_ROOT / "background_hog_features.npz",
    X=X_background,
    y=y_background
)

print("Background features saved.")

In [ ]:
import matplotlib.pyplot as plt

sample_backgrounds = background_manifest.sample(
    min(12, len(background_manifest)),
    random_state=1000
)

fig, axes = plt.subplots(3, 4, figsize=(12, 8))

for ax, (_, row) in zip(
    axes.flat,
    sample_backgrounds.iterrows()
):
    image = Image.open(
        row["image_path"]
    ).convert("RGB")

    crop = image.crop((
        int(row["xmin"]),
        int(row["ymin"]),
        int(row["xmax"]),
        int(row["ymax"])
    ))

    ax.imshow(crop)
    ax.set_title(row["source"], fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:

assert X_positive.shape[1] == X_background.shape[1], (
    "Positive and background feature dimensions do not match."
)

X_all = np.concatenate(
    [X_positive, X_background],
    axis=0
)

y_all = np.concatenate(
    [y_positive, y_background],
    axis=0
)

print("Combined features:", X_all.shape)
print("Combined labels:", y_all.shape)
print("Number of labels represented:", len(np.unique(y_all)))
print("Background label:", BACKGROUND_ID)

In [ ]:
from sklearn.model_selection import train_test_split

# Classes with fewer than two samples cannot be stratified.
class_counts = Counter(y_all)

regular_indices = np.array([
    index
    for index, label in enumerate(y_all)
    if class_counts[label] >= 2
])

rare_indices = np.array([
    index
    for index, label in enumerate(y_all)
    if class_counts[label] < 2
])

train_indices, validation_indices = train_test_split(
    regular_indices,
    test_size=0.20,
    random_state=1000,
    stratify=y_all[regular_indices]
)

# Keep extremely rare samples in training only.
if len(rare_indices) > 0:
    train_indices = np.concatenate(
        [train_indices, rare_indices]
    )

X_train_hog = X_all[train_indices]
y_train_hog = y_all[train_indices]

X_val_hog = X_all[validation_indices]
y_val_hog = y_all[validation_indices]

print("Training features:", X_train_hog.shape)
print("Validation features:", X_val_hog.shape)

In [ ]:
from sklearn.svm import LinearSVC

hog_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    dual="auto",
    max_iter=10000,
    random_state=1000,
    verbose=1
)

hog_svm.fit(
    X_train_hog,
    y_train_hog
)

print("Training complete.")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

y_pred_hog = hog_svm.predict(X_val_hog)

accuracy = accuracy_score(
    y_val_hog,
    y_pred_hog
)

weighted_precision, weighted_recall, weighted_f1, _ = (
    precision_recall_fscore_support(
        y_val_hog,
        y_pred_hog,
        average="weighted",
        zero_division=0
    )
)

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        y_val_hog,
        y_pred_hog,
        average="macro",
        zero_division=0
    )
)

print(f"Accuracy:           {accuracy:.4f}")
print(f"Weighted precision: {weighted_precision:.4f}")
print(f"Weighted recall:    {weighted_recall:.4f}")
print(f"Weighted F1:        {weighted_f1:.4f}")
print(f"Macro F1:           {macro_f1:.4f}")

In [ ]:
background_true = y_val_hog == BACKGROUND_ID
background_pred = y_pred_hog == BACKGROUND_ID

background_accuracy = np.mean(
    background_true == background_pred
)

false_component_rate = np.mean(
    (y_pred_hog != BACKGROUND_ID)
    & background_true
)

print(
    f"Background/component accuracy: "
    f"{background_accuracy:.4f}"
)

print(
    f"Background classified as component: "
    f"{false_component_rate:.4f}"
)

In [ ]:
import joblib

model_path = HOG_ROOT / "hog_linear_svm.joblib"

joblib.dump(
    {
        "model": hog_svm,
        "class_names": primary_class_names,
        "background_id": BACKGROUND_ID,
        "hog_image_size": HOG_IMAGE_SIZE,
        "hog_parameters": {
            "orientations": 9,
            "pixels_per_cell": (8, 8),
            "cells_per_block": (2, 2),
            "block_norm": "L2-Hys"
        }
    },
    model_path
)

print("Model saved to:", model_path)

Convert to grid for yolo (7*7)

In [ ]:
def encode_target(boxes, labels, img_size=448, grid_size=7, num_classes=62):
    target = torch.zeros((grid_size, grid_size, 5 + num_classes))

    cell_size = img_size / grid_size

    for box, label in zip(boxes, labels):
        xmin, ymin, xmax, ymax = box

        x_center = (xmin + xmax) / 2
        y_center = (ymin + ymax) / 2
        w = xmax - xmin
        h = ymax - ymin

        grid_x = int(x_center / cell_size)
        grid_y = int(y_center / cell_size)

        grid_x = min(grid_x, grid_size - 1)
        grid_y = min(grid_y, grid_size - 1)

        x_cell = (x_center - grid_x * cell_size) / cell_size
        y_cell = (y_center - grid_y * cell_size) / cell_size

        w = w / img_size
        h = h / img_size

        target[grid_y, grid_x, 0] = x_cell
        target[grid_y, grid_x, 1] = y_cell
        target[grid_y, grid_x, 2] = torch.sqrt(w)
        target[grid_y, grid_x, 3] = torch.sqrt(h)
        target[grid_y, grid_x, 4] = 1.0
        target[grid_y, grid_x, 5 + label] = 1.0

    return target

In [ ]:
class CircuitDetectionDataset(Dataset):
    def __init__(self, root, split="train", classes_json=None, img_size=448):
        self.root = Path(root)
        self.split = split
        self.image_dir = self.root / "images" / split
        self.ann_dir = self.root / "annotations" / split
        self.img_size = img_size
        self.name = "Yolo"

        with open(classes_json, "r") as f:
            self.class_to_id = json.load(f)

        self.images = sorted([
            p for p in self.image_dir.glob("*")
            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ])

        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        xml_path = self.ann_dir / (img_path.stem + ".xml")

        image = Image.open(img_path).convert("RGB")
        orig_w, orig_h = image.size

        boxes = []
        labels = []

        tree = ET.parse(xml_path)
        xml_root = tree.getroot()

        for obj in xml_root.findall("object"):
            cls_name = obj.find("name").text
            if cls_name not in self.class_to_id:
                continue

            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            # scale boxes to resized image
            xmin = xmin * self.img_size / orig_w
            xmax = xmax * self.img_size / orig_w
            ymin = ymin * self.img_size / orig_h
            ymax = ymax * self.img_size / orig_h

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_id[cls_name])

        image = self.transform(image)

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.long)

        target = encode_target(
            boxes,
            labels,
            img_size=self.img_size,
            grid_size=7,
            num_classes=len(self.class_to_id)
        )

        return image, target

In [ ]:
synthetic_dataset = CircuitDetectionDataset(
    root= synthetic_root,
    split="train",
    classes_json="/content/gdrive/MyDrive/APS360/Project/Data/classes.json",
    img_size=448
)

In [ ]:
train_dataset = CircuitDetectionDataset(output_root, "train", classes_json, img_size=448)
val_dataset = CircuitDetectionDataset(output_root, "val", classes_json, img_size=448)
test_dataset = CircuitDetectionDataset(output_root, "test", classes_json, img_size=448)
train_dataset = ConcatDataset([
    train_dataset,
    synthetic_dataset
])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
images, targets = next(iter(train_loader))

print(images.shape)
print(targets.shape)
print("object cells:", targets[:, :, :, 4].sum())

Training

In [ ]:
def evaluate_detector(model, val_loader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            targets = targets.to(device)

            outputs = model(images)
            loss = yolo_loss(outputs, targets)

            total_loss += loss.item()

    return total_loss / len(val_loader)

In [ ]:

def yolo_loss(pred, target, lambda_coord=5.0, lambda_noobj=1.0):
    # pred/target shape: [B, grid, grid, 67]

    obj_mask = target[..., 4] == 1
    noobj_mask = target[..., 4] == 0

    # bbox loss only for cells with objects
    box_loss = nn.SmoothL1Loss()(
        torch.sigmoid(pred[..., 0:4][obj_mask]),
        target[..., 0:4][obj_mask]
    )
    # objectness loss
    obj_loss = nn.BCEWithLogitsLoss()(pred[..., 4][obj_mask], target[..., 4][obj_mask])
    noobj_loss = nn.BCEWithLogitsLoss()(pred[..., 4][noobj_mask], target[..., 4][noobj_mask])

    # class loss only for cells with objects
    class_loss = nn.CrossEntropyLoss()(
        pred[..., 5:][obj_mask],
        target[..., 5:][obj_mask].argmax(dim=1)
    )

    total_loss = (
        lambda_coord * box_loss
        + obj_loss
        + lambda_noobj * noobj_loss
        + class_loss
    )

    return total_loss

save_dir = Path("/content/gdrive/MyDrive/APS360/Project/checkpoints")
save_dir.mkdir(parents=True, exist_ok=True)
def train_detector(net, train_loader, val_loader, learning_rate=0.001, num_epochs=50):
    optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate, weight_decay=1e-4)

    net.to(device)

    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):

        net.train()
        total_loss = 0.0

        for (images, targets) in train_loader:
            images = images.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()

            outputs = net(images)
            loss = yolo_loss(outputs, targets)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()


        avg_train_loss = total_loss / len(train_loader)
        avg_val_loss = evaluate_detector(net, val_loader,device)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        save_name = (
            f"{net.name}_"
            f"epoch{epoch+1}_"
            f"val{avg_val_loss:.4f}.pt"
        )
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": net.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss
        },
        save_dir / save_name)
        print(
            f"Epoch {epoch+1}: "
            f"Train loss = {avg_train_loss:.4f} | "
            f"Val loss = {avg_val_loss:.4f}"
        )

    return train_losses, val_losses

In [ ]:

class TinyCircuitYOLO(nn.Module):
    def __init__(self, num_classes=62, grid_size=7):
        super().__init__()
        self.grid_size = grid_size
        self.num_classes = num_classes

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 448 -> 224

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 224 -> 112

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 112 -> 56

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 56 -> 28

            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 28 -> 14

            nn.Conv2d(256, 512, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 14 -> 7
        )

        self.head = nn.Conv2d(512, 5 + num_classes, kernel_size=1)

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        x = x.permute(0, 2, 3, 1)
        return x

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Net = TinyCircuitYOLO().to(device)
Net.name = "TinyCircuitYOLO"
train_losses, val_losses = train_detector(Net,train_loader=train_loader,val_loader=val_loader,num_epochs=20)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

Convert the grid to 14 * 14

In [ ]:
def augment_image_and_boxes(image, boxes, img_size=448):
    if random.random() < 0.5:
        image = F.adjust_brightness(image, random.uniform(0.8, 1.2))

    if random.random() < 0.5:
        image = F.adjust_contrast(image, random.uniform(0.8, 1.2))

    return image, boxes

In [ ]:
class CircuitDetectionDataset(Dataset):
    def __init__(self, root, split="train", classes_json=None, img_size=448):
        self.root = Path(root)
        self.split = split
        self.image_dir = self.root / "images" / split
        self.ann_dir = self.root / "annotations" / split
        self.img_size = img_size

        with open(classes_json, "r") as f:
            self.class_to_id = json.load(f)

        self.images = sorted([
            p for p in self.image_dir.glob("*")
            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ])

        self.to_tensor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        xml_path = self.ann_dir / (img_path.stem + ".xml")

        image = Image.open(img_path).convert("RGB")
        orig_w, orig_h = image.size

        boxes = []
        labels = []

        tree = ET.parse(xml_path)
        xml_root = tree.getroot()

        for obj in xml_root.findall("object"):
            cls_name = obj.find("name").text
            if cls_name not in self.class_to_id:
                continue

            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            xmin = xmin * self.img_size / orig_w
            xmax = xmax * self.img_size / orig_w
            ymin = ymin * self.img_size / orig_h
            ymax = ymax * self.img_size / orig_h

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_id[cls_name])

        image = image.resize((self.img_size, self.img_size))

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.long)

        if self.split == "train":
            image, boxes = augment_image_and_boxes(image, boxes, self.img_size)

        image = self.to_tensor(image)

        target = encode_target(
            boxes,
            labels,
            img_size=self.img_size,
            grid_size=14,
            num_classes=len(self.class_to_id)
        )

        return image, target

In [ ]:
train_dataset = CircuitDetectionDataset(output_root, "train", classes_json, img_size=448)
val_dataset = CircuitDetectionDataset(output_root, "val", classes_json, img_size=448)
test_dataset = CircuitDetectionDataset(output_root, "test", classes_json, img_size=448)
synthetic_dataset = CircuitDetectionDataset(
    root= synthetic_root,
    split="train",
    classes_json="/content/gdrive/MyDrive/APS360/Project/Data/classes.json",
    img_size=448
)

train_dataset = ConcatDataset([
    train_dataset,
    synthetic_dataset
])
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
images, targets = next(iter(train_loader))

print(images.shape)
print(targets.shape)
print("object cells:", targets[:, :, :, 4].sum())

In [ ]:
class BetterCircuitYOLO(nn.Module):
    def __init__(self, num_classes=62):
        super().__init__()
        self.name = "BetterCircuitYOLO_grid14"

        def block(in_ch, out_ch, pool=True):
            layers = [
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.LeakyReLU(0.1)
            ]
            if pool:
                layers.append(nn.MaxPool2d(2))
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            block(3, 32),       # 448 -> 224
            block(32, 64),      # 224 -> 112
            block(64, 128),     # 112 -> 56
            block(128, 256),    # 56 -> 28
            block(256, 512),    # 28 -> 14
            block(512, 512, pool=False),

            nn.Dropout2d(0.2)
        )

        self.head = nn.Conv2d(512, 5 + num_classes, kernel_size=1)

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        x = x.permute(0, 2, 3, 1)
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_2 = BetterCircuitYOLO().to(device)
model_2.name = "BetterGrid14_version2"
train_losses, val_losses = train_detector(model_2,train_loader=train_loader,val_loader=val_loader,learning_rate=0.0001,num_epochs=10)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

class ResNetBackboneYOLO(nn.Module):
    def __init__(self, num_classes=62, grid_size=14):
        super().__init__()
        self.name = "ResNet18YOLO_grid14"
        self.grid_size = grid_size

        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        self.backbone = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
        )

        self.adapt = nn.AdaptiveAvgPool2d((grid_size, grid_size))

        self.head = nn.Sequential(
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 5 + num_classes, 1)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.adapt(x)
        x = self.head(x)
        x = x.permute(0, 2, 3, 1)
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_3 = ResNetBackboneYOLO().to(device)
model_3.name = "ResNet_version2"
train_losses, val_losses = train_detector(model_3,train_loader=train_loader,val_loader=val_loader,learning_rate=0.0001,num_epochs=10)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

import matplotlib.pyplot as plt
import matplotlib.patches as patches

def decode_predictions(
    pred,
    img_size=448,
    grid_size=14,
    conf_threshold=0.3
):
    boxes = []
    cell_size = img_size / grid_size
    pred = pred.detach().cpu()

    for gy in range(grid_size):
        for gx in range(grid_size):
            cell = pred[gy, gx]

            objectness = torch.sigmoid(cell[4])

            class_probs = torch.softmax(cell[5:], dim=0)
            class_conf, cls_id = torch.max(class_probs, dim=0)

            score = (objectness * class_conf).item()

            if score >= conf_threshold:
                x_cell = torch.sigmoid(cell[0]).item()
                y_cell = torch.sigmoid(cell[1]).item()

                w = torch.sigmoid(cell[2]).item() ** 2 * img_size
                h = torch.sigmoid(cell[3]).item() ** 2 * img_size

                x_center = (gx + x_cell) * cell_size
                y_center = (gy + y_cell) * cell_size

                xmin = max(0, x_center - w / 2)
                ymin = max(0, y_center - h / 2)
                xmax = min(img_size, x_center + w / 2)
                ymax = min(img_size, y_center + h / 2)

                boxes.append([
                    xmin,
                    ymin,
                    xmax,
                    ymax,
                    score,
                    cls_id.item()
                ])

    return boxes


def decode_target(target, img_size=448, grid_size=14):
    boxes = []
    cell_size = img_size / grid_size
    target = target.cpu()

    for gy in range(grid_size):
        for gx in range(grid_size):
            cell = target[gy, gx]

            if cell[4].item() == 1:
                x_cell = cell[0].item()
                y_cell = cell[1].item()
                w = cell[2].item() ** 2 * img_size
                h = cell[3].item() ** 2 * img_size

                x_center = (gx + x_cell) * cell_size
                y_center = (gy + y_cell) * cell_size

                xmin = x_center - w / 2
                ymin = y_center - h / 2
                xmax = x_center + w / 2
                ymax = y_center + h / 2

                cls_id = torch.argmax(cell[5:]).item()

                boxes.append([xmin, ymin, xmax, ymax, 1.0, cls_id])

    return boxes


def visualize_validation_prediction(model, val_dataset, idx=0, conf_threshold=0.3):
    model.eval()

    image, target = val_dataset[idx]

    with torch.no_grad():
        pred = model(image.unsqueeze(0).to(device))[0]

    pred_boxes = decode_predictions(
        pred,
        img_size=448,
        grid_size=14,
        conf_threshold=conf_threshold
    )

    gt_boxes = decode_target(
        target,
        img_size=448,
        grid_size=14
    )

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(image.permute(1, 2, 0))

    # ground truth: green
    for box in gt_boxes:
        xmin, ymin, xmax, ymax, conf, cls_id = box

        rect = patches.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            fill=False,
            edgecolor="green",
            linewidth=1.5
        )
        ax.add_patch(rect)

    # predictions: red
    for box in pred_boxes:
        xmin, ymin, xmax, ymax, conf, cls_id = box

        rect = patches.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            fill=False,
            edgecolor="red",
            linewidth=1.5
        )
        ax.add_patch(rect)

        ax.text(
            xmin,
            ymin,
            f"{cls_id}:{conf:.2f}",
            color="red",
            fontsize=8
        )

    plt.title(f"Validation image {idx} | Green = GT, Red = Prediction")
    plt.axis("off")
    plt.show()

    print("GT boxes:", len(gt_boxes))
    print("Pred boxes:", len(pred_boxes))

In [ ]:
checkpoint = torch.load(
    "/content/gdrive/MyDrive/APS360/Project/checkpoints/ResNet_version2_epoch5_val0.9808.pt",
    map_location=device
)

model = ResNetBackboneYOLO(num_classes=62)
model.load_state_dict(checkpoint["model_state_dict"])

model.to(device)
model.eval()
test_loss = evaluate_detector(model, test_loader, device)
print("Test loss:", test_loss)

In [ ]:
visualize_validation_prediction(model, val_dataset, idx=1, conf_threshold=0.9)

In [ ]:

def evaluate_classification_f1(model, data_loader, device):
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, targets in data_loader:
            images = images.to(device)
            targets = targets.to(device)

            outputs = model(images)
            # outputs: [B, 14, 14, 5 + num_classes]

            # Only evaluate cells containing a ground-truth object
            obj_mask = targets[..., 4] == 1

            # Predicted class for every grid cell
            pred_classes = outputs[..., 5:].argmax(dim=-1)

            # Ground-truth class from one-hot encoding
            true_classes = targets[..., 5:].argmax(dim=-1)

            # Keep only ground-truth object cells
            y_true.extend(true_classes[obj_mask].cpu().tolist())
            y_pred.extend(pred_classes[obj_mask].cpu().tolist())

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(
        y_true, y_pred, average="weighted", zero_division=0
    )
    recall = recall_score(
        y_true, y_pred, average="weighted", zero_division=0
    )
    weighted_f1 = f1_score(
        y_true, y_pred, average="weighted", zero_division=0
    )
    macro_f1 = f1_score(
        y_true, y_pred, average="macro", zero_division=0
    )

    print(f"Classification Accuracy: {accuracy:.4f}")
    print(f"Weighted Precision: {precision:.4f}")
    print(f"Weighted Recall: {recall:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")

    print("\nClassification report:")
    print(classification_report(
        y_true,
        y_pred,
        zero_division=0
    ))

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "weighted_f1": weighted_f1,
        "macro_f1": macro_f1,
        "y_true": y_true,
        "y_pred": y_pred
    }

In [ ]:
results = evaluate_classification_f1(
    model=model,
    data_loader=test_loader,
    device=device
)

In [ ]:
def compute_ap(recall, precision):
    """
    Compute Pascal VOC-style AP using the area under the
    precision-recall curve.
    """
    recall = np.concatenate(([0.0], recall, [1.0]))
    precision = np.concatenate(([0.0], precision, [0.0]))

    # Precision envelope
    for i in range(len(precision) - 1, 0, -1):
        precision[i - 1] = max(precision[i - 1], precision[i])

    changing_points = np.where(recall[1:] != recall[:-1])[0]

    ap = np.sum(
        (recall[changing_points + 1] - recall[changing_points])
        * precision[changing_points + 1]
    )

    return float(ap)

In [ ]:
def apply_classwise_nms(
    boxes,
    labels,
    scores,
    iou_threshold=0.5
):
    """
    boxes:  Tensor [N, 4] in xyxy format
    labels: Tensor [N]
    scores: Tensor [N]
    """
    if len(boxes) == 0:
        return boxes, labels, scores

    kept_indices = []

    for class_id in labels.unique():
        class_mask = labels == class_id
        class_indices = torch.where(class_mask)[0]

        keep = nms(
            boxes[class_indices],
            scores[class_indices],
            iou_threshold
        )

        kept_indices.append(class_indices[keep])

    kept_indices = torch.cat(kept_indices)

    return (
        boxes[kept_indices],
        labels[kept_indices],
        scores[kept_indices]
    )

In [ ]:
def evaluate_map50(
    model,
    data_loader,
    device,
    num_classes,
    decode_predictions_fn,
    decode_target_fn,
    ap_conf_threshold=0.001,
    metric_conf_threshold=0.90,
    iou_threshold=0.50,
    nms_iou_threshold=0.50
):
    """
    Calculates:
      - AP per class
      - mAP@0.5
      - detection precision
      - detection recall
      - detection F1

    ap_conf_threshold:
        Keep this low so the evaluator can build a complete
        precision-recall curve.

    metric_conf_threshold:
        Confidence threshold used for the single reported
        precision/recall/F1 operating point.
    """

    model.eval()

    # Ground truths:
    # ground_truths[class_id][image_id] = Tensor[N, 4]
    ground_truths = {
        class_id: defaultdict(list)
        for class_id in range(num_classes)
    }

    # Predictions:
    # predictions[class_id] =
    # [(image_id, confidence, box), ...]
    predictions = {
        class_id: []
        for class_id in range(num_classes)
    }

    image_id = 0

    with torch.no_grad():
        for images, targets in data_loader:
            images = images.to(device)
            targets = targets.to(device)

            outputs = model(images)

            batch_size = images.size(0)

            for batch_index in range(batch_size):
                pred_output = outputs[batch_index]
                target = targets[batch_index]

                # Adjust these calls if your decoder arguments differ.
                # Decode predictions
                decoded_pred = decode_predictions_fn(
                    pred_output,
                    conf_threshold=ap_conf_threshold
                )

                if len(decoded_pred) == 0:
                    pred_boxes = torch.empty((0, 4), dtype=torch.float32)
                    pred_scores = torch.empty((0,), dtype=torch.float32)
                    pred_labels = torch.empty((0,), dtype=torch.long)
                else:
                    decoded_pred = torch.tensor(decoded_pred, dtype=torch.float32)

                    pred_boxes = decoded_pred[:, :4]
                    pred_scores = decoded_pred[:, 4]
                    pred_labels = decoded_pred[:, 5].long()


                # Decode ground truth
                decoded_gt = decode_target_fn(target)

                if len(decoded_gt) == 0:
                    gt_boxes = torch.empty((0, 4), dtype=torch.float32)
                    gt_labels = torch.empty((0,), dtype=torch.long)
                else:
                    decoded_gt = torch.tensor(decoded_gt, dtype=torch.float32)

                    gt_boxes = decoded_gt[:, :4]
                    gt_labels = decoded_gt[:, 5].long()

                # Convert everything to tensors on CPU.
                pred_boxes = torch.as_tensor(
                    pred_boxes,
                    dtype=torch.float32
                ).reshape(-1, 4)

                pred_labels = torch.as_tensor(
                    pred_labels,
                    dtype=torch.long
                ).reshape(-1)

                pred_scores = torch.as_tensor(
                    pred_scores,
                    dtype=torch.float32
                ).reshape(-1)

                gt_boxes = torch.as_tensor(
                    gt_boxes,
                    dtype=torch.float32
                ).reshape(-1, 4)

                gt_labels = torch.as_tensor(
                    gt_labels,
                    dtype=torch.long
                ).reshape(-1)

                # Remove duplicate predictions.
                if len(pred_boxes) > 0:
                    pred_boxes, pred_labels, pred_scores = (
                        apply_classwise_nms(
                            pred_boxes,
                            pred_labels,
                            pred_scores,
                            iou_threshold=nms_iou_threshold
                        )
                    )

                # Store ground-truth boxes.
                for box, label in zip(gt_boxes, gt_labels):
                    ground_truths[int(label.item())][image_id].append(
                        box
                    )

                # Store predictions.
                for box, label, score in zip(
                    pred_boxes,
                    pred_labels,
                    pred_scores
                ):
                    predictions[int(label.item())].append(
                        (
                            image_id,
                            float(score.item()),
                            box
                        )
                    )

                image_id += 1

    # Convert ground-truth box lists into tensors.
    for class_id in range(num_classes):
        for img_id in ground_truths[class_id]:
            ground_truths[class_id][img_id] = torch.stack(
                ground_truths[class_id][img_id]
            )

    ap_per_class = {}
    total_tp_at_threshold = 0
    total_fp_at_threshold = 0
    total_gt = 0

    for class_id in range(num_classes):
        class_predictions = predictions[class_id]
        class_ground_truths = ground_truths[class_id]

        num_gt = sum(
            len(boxes)
            for boxes in class_ground_truths.values()
        )

        total_gt += num_gt

        # Classes absent from the test set cannot have meaningful AP.
        if num_gt == 0:
            continue

        # Sort predictions by decreasing confidence.
        class_predictions.sort(
            key=lambda item: item[1],
            reverse=True
        )

        # Track which GT boxes have already been matched.
        matched = {
            img_id: torch.zeros(
                len(boxes),
                dtype=torch.bool
            )
            for img_id, boxes in class_ground_truths.items()
        }

        tp = np.zeros(len(class_predictions))
        fp = np.zeros(len(class_predictions))

        for pred_index, (
            img_id,
            confidence,
            pred_box
        ) in enumerate(class_predictions):

            if img_id not in class_ground_truths:
                fp[pred_index] = 1
                continue

            gt_boxes = class_ground_truths[img_id]

            ious = box_iou(
                pred_box.unsqueeze(0),
                gt_boxes
            ).squeeze(0)

            best_iou, best_gt_index = ious.max(dim=0)

            if (
                best_iou.item() >= iou_threshold
                and not matched[img_id][best_gt_index]
            ):
                tp[pred_index] = 1
                matched[img_id][best_gt_index] = True
            else:
                fp[pred_index] = 1

        cumulative_tp = np.cumsum(tp)
        cumulative_fp = np.cumsum(fp)

        recall = cumulative_tp / max(num_gt, 1)
        precision = cumulative_tp / np.maximum(
            cumulative_tp + cumulative_fp,
            1e-12
        )

        ap_per_class[class_id] = compute_ap(
            recall,
            precision
        )

        # Metrics at one confidence operating point.
        for pred_index, (_, confidence, _) in enumerate(
            class_predictions
        ):
            if confidence >= metric_conf_threshold:
                total_tp_at_threshold += int(tp[pred_index])
                total_fp_at_threshold += int(fp[pred_index])

    evaluated_classes = len(ap_per_class)

    map50 = (
        float(np.mean(list(ap_per_class.values())))
        if evaluated_classes > 0
        else 0.0
    )

    false_negatives = max(
        total_gt - total_tp_at_threshold,
        0
    )

    detection_precision = (
        total_tp_at_threshold
        / max(
            total_tp_at_threshold + total_fp_at_threshold,
            1
        )
    )

    detection_recall = (
        total_tp_at_threshold
        / max(
            total_tp_at_threshold + false_negatives,
            1
        )
    )

    detection_f1 = (
        2
        * detection_precision
        * detection_recall
        / max(
            detection_precision + detection_recall,
            1e-12
        )
    )

    results = {
        "mAP50": map50,
        "precision": detection_precision,
        "recall": detection_recall,
        "f1": detection_f1,
        "true_positives": total_tp_at_threshold,
        "false_positives": total_fp_at_threshold,
        "false_negatives": false_negatives,
        "ap_per_class": ap_per_class,
        "evaluated_classes": evaluated_classes
    }

    print(f"mAP@0.5:           {map50:.4f}")
    print(f"Detection Precision: {detection_precision:.4f}")
    print(f"Detection Recall:    {detection_recall:.4f}")
    print(f"Detection F1:        {detection_f1:.4f}")
    print(f"True Positives:      {total_tp_at_threshold}")
    print(f"False Positives:     {total_fp_at_threshold}")
    print(f"False Negatives:     {false_negatives}")
    print(f"Classes evaluated:   {evaluated_classes}")

    return results

In [ ]:
detection_results = evaluate_map50(
    model=model,
    data_loader=test_loader,
    device=device,
    num_classes=62,
    decode_predictions_fn=decode_predictions,
    decode_target_fn=decode_target,
    ap_conf_threshold=0.001,
    metric_conf_threshold=0.90,
    iou_threshold=0.50,
    nms_iou_threshold=0.50
)

After progress report

YOLOv5 data processing

In [ ]:
SELECTED_CLASSES = {
    "capacitor.unpolarized",
    "diode",
    "diode.light_emitting",
    "diode.zener",
    "gnd",
    "inductor",
    "lamp",
    "operational_amplifier",
    "resistor",
    "switch",
    "transistor.bjt",
    "voltage.ac",
    "voltage.battery",
    "voltage.dc",
}

print("Selected classes:", len(SELECTED_CLASSES))

In [ ]:
from pathlib import Path

CGHD_ROOT = Path(
    "/content/gdrive/MyDrive/APS360/Project/merged_data"
)

SYNTHETIC_ROOT = Path("/content/synthetic_train")

YOLO_ROOT = Path("/content/yolov5_component_data")

sources = [
    {
        "split": "train",
        "source": "cghd",
        "image_dir": CGHD_ROOT / "images" / "train",
        "annotation_dir": CGHD_ROOT / "annotations" / "train",
    },
    {
        "split": "train",
        "source": "synthetic",
        "image_dir": SYNTHETIC_ROOT / "images" / "train",
        "annotation_dir": SYNTHETIC_ROOT / "annotations" / "train",
    },
    {
        "split": "val",
        "source": "cghd",
        "image_dir": CGHD_ROOT / "images" / "val",
        "annotation_dir": CGHD_ROOT / "annotations" / "val",
    },
    {
        "split": "test",
        "source": "cghd",
        "image_dir": CGHD_ROOT / "images" / "test",
        "annotation_dir": CGHD_ROOT / "annotations" / "test",
    },
]

In [ ]:
import shutil
import xml.etree.ElementTree as ET

from collections import Counter
from PIL import Image
import pandas as pd
from tqdm.auto import tqdm

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
}

# Remove only the previously generated YOLO dataset.
if YOLO_ROOT.exists():
    shutil.rmtree(YOLO_ROOT)

for split in ["train", "val", "test"]:
    (YOLO_ROOT / "images" / split).mkdir(
        parents=True,
        exist_ok=True,
    )
    (YOLO_ROOT / "labels" / split).mkdir(
        parents=True,
        exist_ok=True,
    )

In [ ]:
def build_image_map(image_dir):
    image_map = {}

    for image_path in image_dir.iterdir():
        if (
            image_path.is_file()
            and image_path.suffix.lower() in IMAGE_EXTENSIONS
        ):
            image_map[image_path.stem] = image_path

    return image_map

In [ ]:
conversion_records = []
conversion_stats = {}

for source_config in sources:
    split = source_config["split"]
    source_name = source_config["source"]
    image_dir = source_config["image_dir"]
    annotation_dir = source_config["annotation_dir"]

    image_map = build_image_map(image_dir)
    xml_paths = sorted(annotation_dir.glob("*.xml"))

    stats = {
        "images": 0,
        "objects": 0,
        "empty_images": 0,
        "invalid_boxes": 0,
        "missing_images": 0,
        "class_counts": Counter(),
    }

    for xml_path in tqdm(
        xml_paths,
        desc=f"{source_name} {split}",
    ):
        image_path = image_map.get(xml_path.stem)

        if image_path is None:
            stats["missing_images"] += 1
            continue

        try:
            with Image.open(image_path) as image:
                image_width, image_height = image.size

            xml_root = ET.parse(xml_path).getroot()

        except Exception as error:
            print("Failed:", xml_path.name, error)
            continue

        # Prefix prevents name collisions.
        output_stem = f"{source_name}__{xml_path.stem}"

        output_image_path = (
            YOLO_ROOT
            / "images"
            / split
            / f"{output_stem}{image_path.suffix.lower()}"
        )

        output_label_path = (
            YOLO_ROOT
            / "labels"
            / split
            / f"{output_stem}.txt"
        )

        label_lines = []

        for obj in xml_root.findall("object"):
            class_name = obj.findtext("name")

            if class_name is None:
                continue

            class_name = class_name.strip()

            if class_name not in SELECTED_CLASSES:
                continue

            bbox = obj.find("bndbox")

            if bbox is None:
                stats["invalid_boxes"] += 1
                continue

            try:
                xmin = float(bbox.findtext("xmin"))
                ymin = float(bbox.findtext("ymin"))
                xmax = float(bbox.findtext("xmax"))
                ymax = float(bbox.findtext("ymax"))
            except (TypeError, ValueError):
                stats["invalid_boxes"] += 1
                continue

            # Clip the annotation to the image boundary.
            xmin = max(0.0, min(xmin, image_width - 1))
            ymin = max(0.0, min(ymin, image_height - 1))
            xmax = max(0.0, min(xmax, image_width))
            ymax = max(0.0, min(ymax, image_height))

            box_width = xmax - xmin
            box_height = ymax - ymin

            if box_width <= 1 or box_height <= 1:
                stats["invalid_boxes"] += 1
                continue

            x_center = ((xmin + xmax) / 2) / image_width
            y_center = ((ymin + ymax) / 2) / image_height
            normalized_width = box_width / image_width
            normalized_height = box_height / image_height

            # Class 0 means generic "component".
            label_lines.append(
                f"0 "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{normalized_width:.6f} "
                f"{normalized_height:.6f}"
            )

            conversion_records.append({
                "split": split,
                "source": source_name,
                "image_path": str(output_image_path),
                "original_image_path": str(image_path),
                "class_name": class_name,
                "xmin": xmin,
                "ymin": ymin,
                "xmax": xmax,
                "ymax": ymax,
            })

            stats["objects"] += 1
            stats["class_counts"][class_name] += 1

        # Copy the full image into local Colab storage.
        shutil.copy2(
            image_path,
            output_image_path,
        )

        # Empty files are retained as negative images.
        output_label_path.write_text(
            "\n".join(label_lines)
        )

        stats["images"] += 1

        if not label_lines:
            stats["empty_images"] += 1

    conversion_stats[(source_name, split)] = stats

In [ ]:
for (source_name, split), stats in conversion_stats.items():
    print("=" * 60)
    print(f"{source_name.upper()} — {split.upper()}")
    print("Images:", stats["images"])
    print("Selected objects:", stats["objects"])
    print("Images with no selected objects:", stats["empty_images"])
    print("Invalid boxes:", stats["invalid_boxes"])
    print("Missing images:", stats["missing_images"])

    print("\nClass counts:")
    for class_name, count in stats["class_counts"].most_common():
        print(f"  {class_name:30s} {count}")

In [ ]:
component_manifest = pd.DataFrame(conversion_records)

manifest_path = YOLO_ROOT / "component_manifest.csv"
component_manifest.to_csv(manifest_path, index=False)

print("Manifest:", manifest_path)
print("Annotated components:", len(component_manifest))

In [ ]:
data_yaml_path = YOLO_ROOT / "component.yaml"

data_yaml_path.write_text(
    f"""path: {YOLO_ROOT}
train: images/train
val: images/val
test: images/test

nc: 1
names:
  0: component
"""
)

print(data_yaml_path.read_text())

In [ ]:
for split in ["train", "val", "test"]:
    image_count = sum(
        1
        for path in (YOLO_ROOT / "images" / split).iterdir()
        if path.suffix.lower() in IMAGE_EXTENSIONS
    )

    label_count = len(
        list((YOLO_ROOT / "labels" / split).glob("*.txt"))
    )

    object_count = 0

    for label_path in (
        YOLO_ROOT / "labels" / split
    ).glob("*.txt"):
        text = label_path.read_text().strip()

        if text:
            object_count += len(text.splitlines())

    print(
        f"{split:5s}: "
        f"{image_count} images, "
        f"{label_count} labels, "
        f"{object_count} objects"
    )

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

YOLO_ROOT = Path("/content/yolov5_component_data")


def read_yolo_boxes(label_path, image_width, image_height):
    boxes = []

    text = label_path.read_text().strip()

    if not text:
        return boxes

    for line in text.splitlines():
        values = line.split()

        class_id = int(values[0])
        x_center = float(values[1]) * image_width
        y_center = float(values[2]) * image_height
        box_width = float(values[3]) * image_width
        box_height = float(values[4]) * image_height

        xmin = x_center - box_width / 2
        ymin = y_center - box_height / 2
        xmax = x_center + box_width / 2
        ymax = y_center + box_height / 2

        boxes.append([
            class_id,
            xmin,
            ymin,
            xmax,
            ymax
        ])

    return boxes

In [ ]:
split = "train"

image_paths = [
    path
    for path in (YOLO_ROOT / "images" / split).iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

random.seed(1000)
sample_paths = random.sample(image_paths, 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, image_path in zip(axes.flat, sample_paths):
    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    label_path = (
        YOLO_ROOT
        / "labels"
        / split
        / f"{image_path.stem}.txt"
    )

    boxes = read_yolo_boxes(
        label_path,
        width,
        height
    )

    ax.imshow(image)

    for _, xmin, ymin, xmax, ymax in boxes:
        rectangle = patches.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            fill=False,
            linewidth=1.2
        )

        ax.add_patch(rectangle)

    ax.set_title(
        f"{image_path.name[:28]}\n"
        f"{len(boxes)} components",
        fontsize=8
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
%cd /content

!git clone https://github.com/ultralytics/yolov5
%cd /content/yolov5

!pip install -qr requirements.txt

In [ ]:
%cd /content/yolov5

!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 50 \
    --data /content/yolov5_component_data/component.yaml \
    --weights yolov5s.pt \
    --name circuit_component_localizer \
    --patience 10 \
    --workers 2

In [ ]:

source = Path(
    "/content/yolov5/runs/train/"
    "circuit_component_localizer/weights/best.pt"
)

destination = Path(
    "/content/gdrive/MyDrive/APS360/Project/"
    "yolov5_component_localizer_best.pt"
)

shutil.copy2(source, destination)

print("Saved to:", destination)

Test

In [ ]:
%cd /content

!rm -rf /content/yolov5
!git clone https://github.com/ultralytics/yolov5.git
%cd /content/yolov5

# Exact commit shown in your original training log:
!git checkout 6be609f3

!pip install -qr requirements.txt

In [ ]:
%cd /content/yolov5
!git describe --always --long

In [ ]:
%cd /content/yolov5

!python val.py \
    --weights /content/gdrive/MyDrive/APS360/Project/yolov5_component_localizer_best.pt \
    --data /content/yolov5_component_data/component.yaml \
    --task test \
    --img 640 \
    --batch 16 \
    --conf-thres 0.001 \
    --iou-thres 0.6 \
    --name circuit_component_test

In [ ]:
%cd /content/yolov5

!python detect.py \
    --weights /content/gdrive/MyDrive/APS360/Project/yolov5_component_localizer_best.pt \
    --source /content/yolov5_component_data/images/test \
    --img 640 \
    --conf-thres 0.40 \
    --iou-thres 0.50 \
    --max-det 300 \
    --name circuit_component_predictions

In [ ]:
source = Path(
    "/content/yolov5/runs/detect/circuit_component_predictions"
)

destination = Path(
    "/content/gdrive/MyDrive/APS360/Project/"
    "circuit_component_predictions"
)

shutil.copytree(
    source,
    destination,
    dirs_exist_ok=True
)

print("Saved to:", destination)

Classfier

In [ ]:

MANIFEST_PATH = Path(
    "/content/yolov5_component_data/component_manifest.csv"
)

CLASSES_JSON = Path(
    "/content/gdrive/MyDrive/APS360/Project/Data/classes.json"
)

classifier_manifest = pd.read_csv(MANIFEST_PATH)

SELECTED_CLASSES = {
    "capacitor.unpolarized",
    "diode",
    "diode.light_emitting",
    "diode.zener",
    "gnd",
    "inductor",
    "lamp",
    "operational_amplifier",
    "resistor",
    "switch",
    "transistor.bjt",
    "voltage.ac",
    "voltage.battery",
    "voltage.dc",
}

with open(CLASSES_JSON, "r") as file:
    all_classes = json.load(file)

# Preserve the original order from classes.json.
classifier_classes = [
    class_name
    for class_name in all_classes
    if class_name in SELECTED_CLASSES
]

class_to_id = {
    class_name: class_id
    for class_id, class_name in enumerate(classifier_classes)
}

id_to_class = {
    class_id: class_name
    for class_name, class_id in class_to_id.items()
}

classifier_manifest = classifier_manifest[
    classifier_manifest["class_name"].isin(SELECTED_CLASSES)
].copy()

classifier_manifest["class_id"] = (
    classifier_manifest["class_name"].map(class_to_id)
)

print("Classes:", len(classifier_classes))
print(classifier_classes)
print()
print(classifier_manifest.groupby(["split", "source"]).size())

In [ ]:
def expand_box(
    box,
    image_width,
    image_height,
    scale=1.0
):
    xmin, ymin, xmax, ymax = box

    center_x = (xmin + xmax) / 2
    center_y = (ymin + ymax) / 2

    box_width = max(1, xmax - xmin)
    box_height = max(1, ymax - ymin)

    expanded_width = box_width * scale
    expanded_height = box_height * scale

    new_xmin = max(0, center_x - expanded_width / 2)
    new_ymin = max(0, center_y - expanded_height / 2)
    new_xmax = min(image_width, center_x + expanded_width / 2)
    new_ymax = min(image_height, center_y + expanded_height / 2)

    return (
        int(new_xmin),
        int(new_ymin),
        int(new_xmax),
        int(new_ymax)
    )

In [ ]:
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.RandomRotation(8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.90, 1.10)
    ),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

evaluation_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
class CircuitComponentDataset(Dataset):
    def __init__(
        self,
        dataframe,
        transform=None,
        component_scale=1.20,
        context_scale=2.00
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.component_scale = component_scale
        self.context_scale = context_scale

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = Path(row["original_image_path"])
        image = Image.open(image_path).convert("RGB")

        image_width, image_height = image.size

        original_box = (
            float(row["xmin"]),
            float(row["ymin"]),
            float(row["xmax"]),
            float(row["ymax"])
        )

        component_box = expand_box(
            original_box,
            image_width,
            image_height,
            scale=self.component_scale
        )

        context_box = expand_box(
            original_box,
            image_width,
            image_height,
            scale=self.context_scale
        )

        component_crop = image.crop(component_box)
        context_crop = image.crop(context_box)

        if self.transform is not None:
            component_crop = self.transform(component_crop)
            context_crop = self.transform(context_crop)

        label = int(row["class_id"])

        return component_crop, context_crop, label

In [ ]:
train_dataframe = classifier_manifest[
    classifier_manifest["split"] == "train"
].reset_index(drop=True)

val_dataframe = classifier_manifest[
    classifier_manifest["split"] == "val"
].reset_index(drop=True)

test_dataframe = classifier_manifest[
    classifier_manifest["split"] == "test"
].reset_index(drop=True)

train_dataset = CircuitComponentDataset(
    train_dataframe,
    transform=train_transform
)

val_dataset = CircuitComponentDataset(
    val_dataframe,
    transform=evaluation_transform
)

test_dataset = CircuitComponentDataset(
    test_dataframe,
    transform=evaluation_transform
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

In [ ]:

training_labels = train_dataframe["class_id"].to_numpy()

class_counts = np.bincount(
    training_labels,
    minlength=len(classifier_classes)
)

class_weights = 1.0 / np.maximum(class_counts, 1)

sample_weights = class_weights[training_labels]

train_sampler = WeightedRandomSampler(
    weights=torch.tensor(
        sample_weights,
        dtype=torch.double
    ),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    sampler=train_sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
component_crops, context_crops, labels = next(
    iter(train_loader)
)

print("Component crops:", component_crops.shape)
print("Context crops:", context_crops.shape)
print("Labels:", labels.shape)
print("Label range:", labels.min().item(), labels.max().item())

In [ ]:

class FeatureBranch(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 64 -> 32

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32 -> 16

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 16 -> 8

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        x = self.features(x)
        return x.flatten(start_dim=1)

In [ ]:
class DualBranchCircuitClassifier(nn.Module):
    def __init__(self, num_classes=14):
        super().__init__()

        self.component_branch = FeatureBranch()
        self.context_branch = FeatureBranch()

        self.classifier = nn.Sequential(
            nn.Linear(128 + 128, 256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, component_crop, context_crop):
        component_features = self.component_branch(component_crop)
        context_features = self.context_branch(context_crop)

        combined_features = torch.cat(
            [component_features, context_features],
            dim=1
        )

        return self.classifier(combined_features)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = DualBranchCircuitClassifier(
    num_classes=len(classifier_classes)
).to(device)

component_crops = component_crops.to(device)
context_crops = context_crops.to(device)

with torch.no_grad():
    outputs = model(component_crops, context_crops)

print("Device:", device)
print("Model output:", outputs.shape)

In [ ]:
def evaluate_classifier(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for component, context, labels in data_loader:
            component = component.to(device, non_blocking=True)
            context = context.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(component, context)
            loss = criterion(outputs, labels)

            predictions = outputs.argmax(dim=1)

            total_loss += loss.item() * labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    average_loss = total_loss / len(data_loader.dataset)

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="weighted",
            zero_division=0
        )
    )

    _, _, macro_f1, _ = precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return {
        "loss": average_loss,
        "accuracy": accuracy,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,
        "macro_f1": macro_f1
    }

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

checkpoint_path = Path(
    "/content/gdrive/MyDrive/APS360/Project/"
    "dual_branch_classifier_best.pt"
)

In [ ]:
NUM_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 5

best_validation_f1 = 0.0
epochs_without_improvement = 0

training_history = []

for epoch in range(NUM_EPOCHS):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}"
    )

    for component, context, labels in progress_bar:
        component = component.to(device, non_blocking=True)
        context = context.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(component, context)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        predictions = outputs.argmax(dim=1)

        running_loss += loss.item() * labels.size(0)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{correct / total:.4f}"
        )

    training_loss = running_loss / total
    training_accuracy = correct / total

    validation_metrics = evaluate_classifier(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(validation_metrics["weighted_f1"])

    current_learning_rate = optimizer.param_groups[0]["lr"]

    print(
        f"\nEpoch {epoch + 1}: "
        f"Train loss={training_loss:.4f}, "
        f"Train acc={training_accuracy:.4f} | "
        f"Val loss={validation_metrics['loss']:.4f}, "
        f"Val acc={validation_metrics['accuracy']:.4f}, "
        f"Val weighted F1={validation_metrics['weighted_f1']:.4f}, "
        f"Val macro F1={validation_metrics['macro_f1']:.4f}, "
        f"LR={current_learning_rate:.6f}"
    )

    training_history.append({
        "epoch": epoch + 1,
        "train_loss": training_loss,
        "train_accuracy": training_accuracy,
        **{
            f"val_{key}": value
            for key, value in validation_metrics.items()
        }
    })

    if validation_metrics["weighted_f1"] > best_validation_f1:
        best_validation_f1 = validation_metrics["weighted_f1"]
        epochs_without_improvement = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "classifier_classes": classifier_classes,
            "best_validation_f1": best_validation_f1
        }, checkpoint_path)

        print("Saved new best model.")

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint_path = Path(
    "/content/gdrive/MyDrive/APS360/Project/"
    "dual_branch_classifier_best.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

saved_classes = checkpoint["classifier_classes"]

model = DualBranchCircuitClassifier(
    num_classes=len(saved_classes)
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

criterion = nn.CrossEntropyLoss()

print("Loaded checkpoint:", checkpoint_path)
print("Best validation F1:", checkpoint["best_validation_f1"])
print("Classes:", saved_classes)

In [ ]:

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("CGHD test samples:", len(test_dataset))

In [ ]:
cghd_classifier_test_results = evaluate_classifier(
    model,
    test_loader,
    criterion,
    device
)

print("\nCGHD dual-branch classifier test results:")

for metric, value in cghd_classifier_test_results.items():
    print(f"{metric}: {value:.4f}")

Overall Pipeline Testing

In [ ]:
cghd_test_ground_truth = test_dataframe.copy()

# Use the original filename stem as the image identifier.
cghd_test_ground_truth["image_id"] = (
    cghd_test_ground_truth["original_image_path"]
    .apply(lambda path: Path(path).stem)
)

cghd_test_images = (
    cghd_test_ground_truth[
        ["image_id", "original_image_path"]
    ]
    .drop_duplicates()
    .rename(columns={
        "original_image_path": "image_path"
    })
    .reset_index(drop=True)
)

print("CGHD test diagrams:", len(cghd_test_images))
print("CGHD test components:", len(cghd_test_ground_truth))

In [ ]:

YOLO_WEIGHTS = Path(
    "/content/gdrive/MyDrive/APS360/Project/"
    "yolov5_component_localizer_best.pt"
)

yolo_model = torch.hub.load(
    "/content/yolov5",
    "custom",
    path=str(YOLO_WEIGHTS),
    source="local"
)

yolo_model.to(device)

yolo_model.conf = 0.001
yolo_model.iou = 0.50
yolo_model.max_det = 300

yolo_model.eval()
model.eval()

print("YOLOv5 localizer loaded.")

In [ ]:

PIPELINE_BATCH_SIZE = 8
all_cghd_predictions = []

image_records = cghd_test_images.to_dict("records")

for start_index in tqdm(
    range(0, len(image_records), PIPELINE_BATCH_SIZE),
    desc="CGHD full-pipeline inference"
):
    batch_records = image_records[
        start_index:
        start_index + PIPELINE_BATCH_SIZE
    ]

    batch_paths = [
        record["image_path"]
        for record in batch_records
    ]

    # YOLO localization on complete diagrams
    batch_yolo_results = yolo_model(
        batch_paths,
        size=640
    )

    for record, detection_tensor in zip(
        batch_records,
        batch_yolo_results.xyxy
    ):
        image = Image.open(
            record["image_path"]
        ).convert("RGB")

        image_width, image_height = image.size

        component_tensors = []
        context_tensors = []
        detection_metadata = []

        for detection in detection_tensor.detach().cpu():
            xmin, ymin, xmax, ymax, yolo_confidence, _ = (
                detection.tolist()
            )

            xmin = max(0.0, min(xmin, image_width))
            xmax = max(0.0, min(xmax, image_width))
            ymin = max(0.0, min(ymin, image_height))
            ymax = max(0.0, min(ymax, image_height))

            if xmax <= xmin or ymax <= ymin:
                continue

            detected_box = (
                xmin,
                ymin,
                xmax,
                ymax
            )

            component_box = expand_box(
                detected_box,
                image_width,
                image_height,
                scale=1.20
            )

            context_box = expand_box(
                detected_box,
                image_width,
                image_height,
                scale=2.00
            )

            component_crop = image.crop(component_box)
            context_crop = image.crop(context_box)

            component_tensors.append(
                evaluation_transform(component_crop)
            )

            context_tensors.append(
                evaluation_transform(context_crop)
            )

            detection_metadata.append({
                "image_id": record["image_id"],
                "image_path": record["image_path"],
                "xmin": xmin,
                "ymin": ymin,
                "xmax": xmax,
                "ymax": ymax,
                "detection_confidence": float(
                    yolo_confidence
                )
            })

        if not component_tensors:
            continue

        component_batch = torch.stack(
            component_tensors
        ).to(device)

        context_batch = torch.stack(
            context_tensors
        ).to(device)

        with torch.no_grad():
            outputs = model(
                component_batch,
                context_batch
            )

            probabilities = torch.softmax(
                outputs,
                dim=1
            )

            class_confidences, predicted_ids = (
                probabilities.max(dim=1)
            )

        for metadata, predicted_id, class_confidence in zip(
            detection_metadata,
            predicted_ids.cpu().numpy(),
            class_confidences.cpu().numpy()
        ):
            predicted_id = int(predicted_id)
            class_confidence = float(class_confidence)

            all_cghd_predictions.append({
                **metadata,
                "predicted_class_id": predicted_id,
                "predicted_class": (
                    classifier_classes[predicted_id]
                ),
                "classification_confidence": (
                    class_confidence
                ),
                "final_confidence": (
                    metadata["detection_confidence"]
                    * class_confidence
                )
            })

In [ ]:
cghd_pipeline_predictions = pd.DataFrame(
    all_cghd_predictions
)

CGHD_PREDICTION_PATH = Path(
    "/content/gdrive/MyDrive/APS360/Project/"
    "cghd_test_full_pipeline_predictions.csv"
)

cghd_pipeline_predictions.to_csv(
    CGHD_PREDICTION_PATH,
    index=False
)

print("Predictions:", len(cghd_pipeline_predictions))
print("Saved:", CGHD_PREDICTION_PATH)

In [ ]:


def calculate_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    intersection_x1 = max(ax1, bx1)
    intersection_y1 = max(ay1, by1)
    intersection_x2 = min(ax2, bx2)
    intersection_y2 = min(ay2, by2)

    intersection_width = max(
        0.0,
        intersection_x2 - intersection_x1
    )
    intersection_height = max(
        0.0,
        intersection_y2 - intersection_y1
    )

    intersection_area = (
        intersection_width * intersection_height
    )

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union_area = area_a + area_b - intersection_area

    if union_area <= 0:
        return 0.0

    return intersection_area / union_area

In [ ]:
def evaluate_pipeline_at_threshold(
    prediction_dataframe,
    ground_truth_dataframe,
    confidence_threshold,
    iou_threshold=0.5
):
    predictions = prediction_dataframe[
        prediction_dataframe["final_confidence"]
        >= confidence_threshold
    ].copy()

    predictions = predictions.sort_values(
        "final_confidence",
        ascending=False
    )

    matched_ground_truth = set()
    result_records = []

    for prediction_index, prediction in predictions.iterrows():
        candidates = ground_truth_dataframe[
            (
                ground_truth_dataframe["image_id"]
                == prediction["image_id"]
            )
            &
            (
                ground_truth_dataframe["class_name"]
                == prediction["predicted_class"]
            )
        ]

        best_iou = 0.0
        best_ground_truth_index = None

        prediction_box = (
            prediction["xmin"],
            prediction["ymin"],
            prediction["xmax"],
            prediction["ymax"]
        )

        for ground_truth_index, ground_truth in candidates.iterrows():
            if ground_truth_index in matched_ground_truth:
                continue

            ground_truth_box = (
                ground_truth["xmin"],
                ground_truth["ymin"],
                ground_truth["xmax"],
                ground_truth["ymax"]
            )

            current_iou = calculate_iou(
                prediction_box,
                ground_truth_box
            )

            if current_iou > best_iou:
                best_iou = current_iou
                best_ground_truth_index = ground_truth_index

        is_true_positive = (
            best_ground_truth_index is not None
            and best_iou >= iou_threshold
        )

        if is_true_positive:
            matched_ground_truth.add(
                best_ground_truth_index
            )

        result_records.append({
            "prediction_index": prediction_index,
            "image_id": prediction["image_id"],
            "predicted_class": prediction["predicted_class"],
            "confidence": prediction["final_confidence"],
            "best_iou": best_iou,
            "true_positive": int(is_true_positive),
            "false_positive": int(not is_true_positive)
        })

    true_positives = sum(
        record["true_positive"]
        for record in result_records
    )

    false_positives = sum(
        record["false_positive"]
        for record in result_records
    )

    false_negatives = (
        len(ground_truth_dataframe)
        - len(matched_ground_truth)
    )

    precision = (
        true_positives
        / max(true_positives + false_positives, 1)
    )

    recall = (
        true_positives
        / max(true_positives + false_negatives, 1)
    )

    f1_score = (
        2 * precision * recall
        / max(precision + recall, 1e-12)
    )

    return {
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "precision": precision,
        "recall": recall,
        "f1": f1_score,
        "matched_predictions": pd.DataFrame(result_records)
    }

In [ ]:
CGHD_OPERATING_THRESHOLD = 0.20

cghd_detection_results = evaluate_pipeline_at_threshold(
    prediction_dataframe=cghd_pipeline_predictions,
    ground_truth_dataframe=cghd_test_ground_truth,
    confidence_threshold=CGHD_OPERATING_THRESHOLD,
    iou_threshold=0.50
)

for metric in [
    "true_positives",
    "false_positives",
    "false_negatives",
    "precision",
    "recall",
    "f1"
]:
    value = cghd_detection_results[metric]

    if isinstance(value, float):
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value}")

In [ ]:
def calculate_ap(recall, precision):
    recall = np.concatenate(([0.0], recall, [1.0]))
    precision = np.concatenate(([0.0], precision, [0.0]))

    for index in range(
        len(precision) - 2,
        -1,
        -1
    ):
        precision[index] = max(
            precision[index],
            precision[index + 1]
        )

    changing_points = np.where(
        recall[1:] != recall[:-1]
    )[0]

    average_precision = np.sum(
        (recall[changing_points + 1]
         - recall[changing_points])
        * precision[changing_points + 1]
    )

    return average_precision

In [ ]:
def calculate_class_ap(
    prediction_dataframe,
    ground_truth_dataframe,
    class_name,
    iou_threshold=0.5
):
    class_ground_truth = ground_truth_dataframe[
        ground_truth_dataframe["class_name"] == class_name
    ]

    class_predictions = prediction_dataframe[
        prediction_dataframe["predicted_class"] == class_name
    ].sort_values(
        "final_confidence",
        ascending=False
    )

    number_of_ground_truth = len(class_ground_truth)

    if number_of_ground_truth == 0:
        return None

    matched_ground_truth = set()
    true_positives = []
    false_positives = []

    for _, prediction in class_predictions.iterrows():
        candidates = class_ground_truth[
            class_ground_truth["image_id"]
            == prediction["image_id"]
        ]

        prediction_box = (
            prediction["xmin"],
            prediction["ymin"],
            prediction["xmax"],
            prediction["ymax"]
        )

        best_iou = 0.0
        best_index = None

        for ground_truth_index, ground_truth in candidates.iterrows():
            if ground_truth_index in matched_ground_truth:
                continue

            ground_truth_box = (
                ground_truth["xmin"],
                ground_truth["ymin"],
                ground_truth["xmax"],
                ground_truth["ymax"]
            )

            current_iou = calculate_iou(
                prediction_box,
                ground_truth_box
            )

            if current_iou > best_iou:
                best_iou = current_iou
                best_index = ground_truth_index

        if best_index is not None and best_iou >= iou_threshold:
            matched_ground_truth.add(best_index)
            true_positives.append(1)
            false_positives.append(0)
        else:
            true_positives.append(0)
            false_positives.append(1)

    if len(class_predictions) == 0:
        return 0.0

    cumulative_tp = np.cumsum(true_positives)
    cumulative_fp = np.cumsum(false_positives)

    recall = cumulative_tp / number_of_ground_truth

    precision = cumulative_tp / np.maximum(
        cumulative_tp + cumulative_fp,
        1
    )

    return calculate_ap(recall, precision)

In [ ]:
cghd_present_classes = sorted(
    cghd_test_ground_truth["class_name"].unique()
)

cghd_class_ap_results = {}

for class_name in cghd_present_classes:
    cghd_class_ap_results[class_name] = (
        calculate_class_ap(
            prediction_dataframe=cghd_pipeline_predictions,
            ground_truth_dataframe=cghd_test_ground_truth,
            class_name=class_name,
            iou_threshold=0.50
        )
    )

cghd_class_ap_dataframe = pd.DataFrame({
    "class_name": cghd_class_ap_results.keys(),
    "AP@0.5": cghd_class_ap_results.values()
}).sort_values(
    "AP@0.5",
    ascending=False
)

cghd_map50 = (
    cghd_class_ap_dataframe["AP@0.5"].mean()
)

print(cghd_class_ap_dataframe.to_string(index=False))
print(
    f"\nCGHD end-to-end class-aware "
    f"mAP@0.5: {cghd_map50:.4f}"
)

New data set

In [ ]:

DIGITIZE_ROOT = Path("/content/digitize_hcd_external_test")

DIGITIZE_ROOT.mkdir(parents=True, exist_ok=True)

# Extract only when the folder is empty.
if not any(DIGITIZE_ROOT.iterdir()):
    with zipfile.ZipFile("/content/gdrive/MyDrive/APS360/Project/Digitize-HCD Dataset.zip", "r") as zip_file:
        zip_file.extractall(DIGITIZE_ROOT)

    print("Dataset extracted.")
else:
    print("Dataset already extracted.")

print("Extraction path:", DIGITIZE_ROOT)

In [ ]:
digitize_to_classifier = {
    "BJT-NPN": "transistor.bjt",
    "BJT-PNP": "transistor.bjt",
    "Capacitor": "capacitor.unpolarized",
    "Diode": "diode",
    "GND": "gnd",
    "Inductor": "inductor",
    "Op-Amp": "operational_amplifier",
    "Resistor": "resistor",
    "V-AC": "voltage.ac",
    "V-DC": "voltage.dc",
    "Zener Diode": "diode.zener",
}

In [ ]:

DIGITIZE_BASE = Path(
    "/content/digitize_hcd_external_test/"
    "Digitize-HCD Dataset/"
    "Component Symbol and Text Label Data"
)

ANNOTATION_PATH = DIGITIZE_BASE / "component_annotations.json"
IMAGE_DIR = DIGITIZE_BASE / "Circuit Diagram Images"

with open(ANNOTATION_PATH, "r", encoding="utf-8") as file:
    digitize_data = json.load(file)

print("Images:", len(digitize_data["images"]))
print("Annotations:", len(digitize_data["annotations"]))
print("Categories:", len(digitize_data["categories"]))

In [ ]:
category_id_to_name = {
    category["id"]: category["name"]
    for category in digitize_data["categories"]
}

image_id_to_info = {
    image["id"]: image
    for image in digitize_data["images"]
}

print(category_id_to_name)

In [ ]:
external_records = []
skipped_categories = Counter()
missing_images = 0
invalid_boxes = 0

for annotation in digitize_data["annotations"]:
    source_class = category_id_to_name[
        annotation["category_id"]
    ]

    if source_class not in digitize_to_classifier:
        skipped_categories[source_class] += 1
        continue

    target_class = digitize_to_classifier[source_class]

    image_info = image_id_to_info[annotation["image_id"]]
    image_path = IMAGE_DIR / image_info["file_name"]

    if not image_path.exists():
        missing_images += 1
        continue

    xmin, ymin, box_width, box_height = annotation["bbox"]

    xmax = xmin + box_width
    ymax = ymin + box_height

    image_width = image_info.get("width")
    image_height = image_info.get("height")

    if image_width is not None:
        xmin = max(0.0, min(float(xmin), float(image_width)))
        xmax = max(0.0, min(float(xmax), float(image_width)))

    if image_height is not None:
        ymin = max(0.0, min(float(ymin), float(image_height)))
        ymax = max(0.0, min(float(ymax), float(image_height)))

    if xmax <= xmin or ymax <= ymin:
        invalid_boxes += 1
        continue

    external_records.append({
        "split": "external_test",
        "source": "Digitize-HCD",
        "original_image_path": str(image_path),
        "class_name": target_class,
        "class_id": class_to_id[target_class],
        "xmin": float(xmin),
        "ymin": float(ymin),
        "xmax": float(xmax),
        "ymax": float(ymax),
        "digitize_class": source_class,
        "image_id": annotation["image_id"],
        "annotation_id": annotation.get("id")
    })

new_test_dataframe = pd.DataFrame(external_records)

print("External test samples:", len(new_test_dataframe))
print("Mapped target classes:", new_test_dataframe["class_name"].nunique())
print("Missing images:", missing_images)
print("Invalid boxes:", invalid_boxes)

print("\nSkipped categories:")
print(skipped_categories)

In [ ]:
external_distribution = (
    new_test_dataframe["class_name"]
    .value_counts()
    .reindex(classifier_classes, fill_value=0)
)

print(external_distribution)

In [ ]:
EXTERNAL_MANIFEST_PATH = Path(
    "/content/gdrive/MyDrive/APS360/Project/"
    "digitize_hcd_external_test_manifest.csv"
)

new_test_dataframe.to_csv(
    EXTERNAL_MANIFEST_PATH,
    index=False
)

print("Saved:", EXTERNAL_MANIFEST_PATH)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

samples = new_test_dataframe.sample(
    min(12, len(new_test_dataframe)),
    random_state=1000
)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))

for ax, (_, row) in zip(axes.flat, samples.iterrows()):
    image = Image.open(
        row["original_image_path"]
    ).convert("RGB")

    crop = image.crop((
        int(row["xmin"]),
        int(row["ymin"]),
        int(row["xmax"]),
        int(row["ymax"])
    ))

    ax.imshow(crop)
    ax.set_title(
        f'{row["digitize_class"]}\n'
        f'→ {row["class_name"]}',
        fontsize=8
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:

new_test_dataset = CircuitComponentDataset(
    new_test_dataframe,
    transform=evaluation_transform
)

new_test_loader = DataLoader(
    new_test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("External test samples:", len(new_test_dataset))

In [ ]:
external_results = evaluate_classifier(
    model,
    new_test_loader,
    criterion,
    device
)

print("\nDigitize-HCD classification results:")

for metric, value in external_results.items():
    print(f"{metric}: {value:.4f}")

In [ ]:

all_labels = []
all_predictions = []
all_probabilities = []

model.eval()

with torch.no_grad():
    for component, context, labels in new_test_loader:
        component = component.to(device)
        context = context.to(device)

        outputs = model(component, context)
        probabilities = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities.cpu().numpy())

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)

print("Evaluated samples:", len(all_labels))

In [ ]:

present_class_ids = sorted(np.unique(all_labels))

present_class_names = [
    classifier_classes[class_id]
    for class_id in present_class_ids
]

report = classification_report(
    all_labels,
    all_predictions,
    labels=present_class_ids,
    target_names=present_class_names,
    digits=4,
    zero_division=0
)

print(report)

In [ ]:

report_dict = classification_report(
    all_labels,
    all_predictions,
    labels=present_class_ids,
    target_names=present_class_names,
    output_dict=True,
    zero_division=0
)

per_class_results = pd.DataFrame(report_dict).transpose()
per_class_results

In [ ]:
cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=present_class_ids
)

fig, ax = plt.subplots(figsize=(11, 9))

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=present_class_names
)

display.plot(
    ax=ax,
    xticks_rotation=45,
    values_format="d"
)

plt.title("Digitize-HCD Class Recognition Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
cm_normalized = confusion_matrix(
    all_labels,
    all_predictions,
    labels=present_class_ids,
    normalize="true"
)

fig, ax = plt.subplots(figsize=(11, 9))

display = ConfusionMatrixDisplay(
    confusion_matrix=cm_normalized,
    display_labels=present_class_names
)

display.plot(
    ax=ax,
    xticks_rotation=45,
    values_format=".2f"
)

plt.title("Normalized Digitize-HCD Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
new_test_dataframe = new_test_dataframe.copy()

new_test_dataframe["true_class_id"] = all_labels
new_test_dataframe["predicted_class_id"] = all_predictions

new_test_dataframe["predicted_class"] = [
    classifier_classes[class_id]
    for class_id in all_predictions
]

new_test_dataframe["confidence"] = (
    all_probabilities.max(axis=1)
)

new_test_dataframe["correct"] = (
    new_test_dataframe["true_class_id"]
    == new_test_dataframe["predicted_class_id"]
)

prediction_path = Path(
    "/content/gdrive/MyDrive/APS360/Project/"
    "digitize_hcd_classification_predictions.csv"
)

new_test_dataframe.to_csv(
    prediction_path,
    index=False
)

print("Saved predictions:", prediction_path)
print("Correct:", new_test_dataframe["correct"].sum())
print("Incorrect:", (~new_test_dataframe["correct"]).sum())

In [ ]:

def show_classification_samples(
    dataframe,
    correct=True,
    number=12,
    random_state=1000
):
    selected = dataframe[
        dataframe["correct"] == correct
    ]

    if len(selected) == 0:
        print("No samples available.")
        return

    selected = selected.sample(
        min(number, len(selected)),
        random_state=random_state
    )

    rows = 3
    columns = 4

    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(12, 9)
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, (_, row) in zip(
        axes.flat,
        selected.iterrows()
    ):
        image = Image.open(
            row["original_image_path"]
        ).convert("RGB")

        crop = image.crop((
            int(row["xmin"]),
            int(row["ymin"]),
            int(row["xmax"]),
            int(row["ymax"])
        ))

        ax.imshow(crop)
        ax.set_title(
            f'True: {row["class_name"]}\n'
            f'Pred: {row["predicted_class"]}\n'
            f'Conf: {row["confidence"]:.2f}',
            fontsize=8
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_classification_samples(
    new_test_dataframe,
    correct=True
)

In [ ]:
show_classification_samples(
    new_test_dataframe,
    correct=False
)

In [ ]:

YOLO_WEIGHTS = Path(
    "/content/gdrive/MyDrive/APS360/Project/"
    "yolov5_component_localizer_best.pt"
)

yolo_model = torch.hub.load(
    "/content/yolov5",
    "custom",
    path=str(YOLO_WEIGHTS),
    source="local"
)

yolo_model.to(device)
yolo_model.eval()

# Detection settings
yolo_model.conf = 0.25
yolo_model.iou = 0.50
yolo_model.max_det = 300

print("YOLOv5 localizer loaded.")

In [ ]:
CLASSIFIER_WEIGHTS = Path(
    "/content/gdrive/MyDrive/APS360/Project/"
    "dual_branch_classifier_best.pt"
)

checkpoint = torch.load(
    CLASSIFIER_WEIGHTS,
    map_location=device
)

classifier_classes = checkpoint["classifier_classes"]

classifier_model = DualBranchCircuitClassifier(
    num_classes=len(classifier_classes)
).to(device)

classifier_model.load_state_dict(
    checkpoint["model_state_dict"]
)

classifier_model.eval()

print("Classifier loaded.")
print("Classes:", classifier_classes)

In [ ]:


def run_full_pipeline(
    image_path,
    yolo_model,
    classifier_model,
    classifier_classes,
    transform,
    device,
    component_scale=1.20,
    context_scale=2.00
):
    image_path = Path(image_path)
    image = Image.open(image_path).convert("RGB")

    image_width, image_height = image.size

    # YOLO localization
    yolo_results = yolo_model(
        str(image_path),
        size=640
    )

    detections = yolo_results.xyxy[0].detach().cpu()

    if len(detections) == 0:
        return []

    component_tensors = []
    context_tensors = []
    detection_information = []

    for detection in detections:
        xmin, ymin, xmax, ymax, detection_confidence, _ = (
            detection.tolist()
        )

        xmin = max(0.0, min(xmin, image_width))
        xmax = max(0.0, min(xmax, image_width))
        ymin = max(0.0, min(ymin, image_height))
        ymax = max(0.0, min(ymax, image_height))

        if xmax <= xmin or ymax <= ymin:
            continue

        detected_box = (
            xmin,
            ymin,
            xmax,
            ymax
        )

        component_box = expand_box(
            detected_box,
            image_width,
            image_height,
            scale=component_scale
        )

        context_box = expand_box(
            detected_box,
            image_width,
            image_height,
            scale=context_scale
        )

        component_crop = image.crop(component_box)
        context_crop = image.crop(context_box)

        component_tensors.append(
            transform(component_crop)
        )

        context_tensors.append(
            transform(context_crop)
        )

        detection_information.append({
            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax,
            "detection_confidence": detection_confidence
        })

    if len(component_tensors) == 0:
        return []

    component_batch = torch.stack(
        component_tensors
    ).to(device)

    context_batch = torch.stack(
        context_tensors
    ).to(device)

    # Class recognition
    with torch.no_grad():
        logits = classifier_model(
            component_batch,
            context_batch
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        class_confidences, predicted_ids = (
            probabilities.max(dim=1)
        )

    predictions = []

    for information, predicted_id, class_confidence in zip(
        detection_information,
        predicted_ids.cpu().numpy(),
        class_confidences.cpu().numpy()
    ):
        predicted_id = int(predicted_id)

        # Combined confidence for ranking final predictions
        final_confidence = (
            information["detection_confidence"]
            * float(class_confidence)
        )

        predictions.append({
            **information,
            "predicted_class_id": predicted_id,
            "predicted_class": classifier_classes[predicted_id],
            "classification_confidence": float(class_confidence),
            "final_confidence": float(final_confidence)
        })

    return predictions

In [ ]:
test_image_info = digitize_data["images"][0]

test_image_path = (
    IMAGE_DIR / test_image_info["file_name"]
)

test_predictions = run_full_pipeline(
    image_path=test_image_path,
    yolo_model=yolo_model,
    classifier_model=classifier_model,
    classifier_classes=classifier_classes,
    transform=evaluation_transform,
    device=device
)

print("Image:", test_image_path)
print("Detected components:", len(test_predictions))

for prediction in test_predictions[:10]:
    print(
        prediction["predicted_class"],
        f'YOLO={prediction["detection_confidence"]:.3f}',
        f'Class={prediction["classification_confidence"]:.3f}',
        f'Final={prediction["final_confidence"]:.3f}'
    )

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


def show_full_pipeline_predictions(
    image_path,
    predictions,
    minimum_final_confidence=0.20
):
    image = Image.open(image_path).convert("RGB")

    fig, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(image)

    displayed = 0

    for prediction in predictions:
        if (
            prediction["final_confidence"]
            < minimum_final_confidence
        ):
            continue

        xmin = prediction["xmin"]
        ymin = prediction["ymin"]
        width = prediction["xmax"] - xmin
        height = prediction["ymax"] - ymin

        rectangle = Rectangle(
            (xmin, ymin),
            width,
            height,
            fill=False,
            linewidth=1.5
        )

        ax.add_patch(rectangle)

        label = (
            f'{prediction["predicted_class"]} '
            f'{prediction["final_confidence"]:.2f}'
        )

        ax.text(
            xmin,
            max(0, ymin - 4),
            label,
            fontsize=7,
            bbox={
                "facecolor": "white",
                "alpha": 0.75,
                "pad": 1
            }
        )

        displayed += 1

    ax.axis("off")
    ax.set_title(
        f"YOLOv5 + Dual-Branch Classifier "
        f"({displayed} predictions)"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
show_full_pipeline_predictions(
    test_image_path,
    test_predictions,
    minimum_final_confidence=0.20
)

In [ ]:

external_image_records = []

for image_info in digitize_data["images"]:
    image_path = IMAGE_DIR / image_info["file_name"]

    if image_path.exists():
        external_image_records.append({
            "image_id": image_info["id"],
            "image_path": str(image_path),
            "file_name": image_info["file_name"]
        })

print("External images:", len(external_image_records))

In [ ]:
PIPELINE_BATCH_SIZE = 8

all_pipeline_predictions = []

for start_index in tqdm(
    range(0, len(external_image_records), PIPELINE_BATCH_SIZE)
):
    batch_records = external_image_records[
        start_index:
        start_index + PIPELINE_BATCH_SIZE
    ]

    batch_paths = [
        record["image_path"]
        for record in batch_records
    ]

    # Run YOLO on a batch of complete diagrams
    batch_yolo_results = yolo_model(
        batch_paths,
        size=640
    )

    for record, detection_tensor in zip(
        batch_records,
        batch_yolo_results.xyxy
    ):
        image = Image.open(
            record["image_path"]
        ).convert("RGB")

        image_width, image_height = image.size

        component_tensors = []
        context_tensors = []
        metadata = []

        for detection in detection_tensor.detach().cpu():
            xmin, ymin, xmax, ymax, yolo_confidence, _ = (
                detection.tolist()
            )

            xmin = max(0.0, min(xmin, image_width))
            xmax = max(0.0, min(xmax, image_width))
            ymin = max(0.0, min(ymin, image_height))
            ymax = max(0.0, min(ymax, image_height))

            if xmax <= xmin or ymax <= ymin:
                continue

            detected_box = (
                xmin,
                ymin,
                xmax,
                ymax
            )

            component_box = expand_box(
                detected_box,
                image_width,
                image_height,
                scale=1.20
            )

            context_box = expand_box(
                detected_box,
                image_width,
                image_height,
                scale=2.00
            )

            component_tensors.append(
                evaluation_transform(
                    image.crop(component_box)
                )
            )

            context_tensors.append(
                evaluation_transform(
                    image.crop(context_box)
                )
            )

            metadata.append({
                "image_id": record["image_id"],
                "image_path": record["image_path"],
                "file_name": record["file_name"],
                "xmin": xmin,
                "ymin": ymin,
                "xmax": xmax,
                "ymax": ymax,
                "detection_confidence": yolo_confidence
            })

        if len(component_tensors) == 0:
            continue

        component_batch = torch.stack(
            component_tensors
        ).to(device)

        context_batch = torch.stack(
            context_tensors
        ).to(device)

        with torch.no_grad():
            logits = classifier_model(
                component_batch,
                context_batch
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            class_confidences, predicted_ids = (
                probabilities.max(dim=1)
            )

        for information, predicted_id, class_confidence in zip(
            metadata,
            predicted_ids.cpu().numpy(),
            class_confidences.cpu().numpy()
        ):
            predicted_id = int(predicted_id)

            all_pipeline_predictions.append({
                **information,
                "predicted_class_id": predicted_id,
                "predicted_class": (
                    classifier_classes[predicted_id]
                ),
                "classification_confidence": float(
                    class_confidence
                ),
                "final_confidence": float(
                    information["detection_confidence"]
                    * class_confidence
                )
            })

In [ ]:
pipeline_predictions_dataframe = pd.DataFrame(
    all_pipeline_predictions
)

PIPELINE_PREDICTIONS_PATH = Path(
    "/content/gdrive/MyDrive/APS360/Project/"
    "digitize_hcd_full_pipeline_predictions.csv"
)

pipeline_predictions_dataframe.to_csv(
    PIPELINE_PREDICTIONS_PATH,
    index=False
)

print("Predictions:", len(pipeline_predictions_dataframe))
print("Saved:", PIPELINE_PREDICTIONS_PATH)

In [ ]:
print(
    pipeline_predictions_dataframe[
        "predicted_class"
    ].value_counts()
)

In [ ]:


def calculate_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    intersection_x1 = max(ax1, bx1)
    intersection_y1 = max(ay1, by1)
    intersection_x2 = min(ax2, bx2)
    intersection_y2 = min(ay2, by2)

    intersection_width = max(
        0.0,
        intersection_x2 - intersection_x1
    )
    intersection_height = max(
        0.0,
        intersection_y2 - intersection_y1
    )

    intersection_area = (
        intersection_width * intersection_height
    )

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union_area = area_a + area_b - intersection_area

    if union_area <= 0:
        return 0.0

    return intersection_area / union_area

In [ ]:
def evaluate_pipeline_at_threshold(
    prediction_dataframe,
    ground_truth_dataframe,
    confidence_threshold,
    iou_threshold=0.5
):
    predictions = prediction_dataframe[
        prediction_dataframe["final_confidence"]
        >= confidence_threshold
    ].copy()

    predictions = predictions.sort_values(
        "final_confidence",
        ascending=False
    )

    matched_ground_truth = set()
    result_records = []

    for prediction_index, prediction in predictions.iterrows():
        candidates = ground_truth_dataframe[
            (
                ground_truth_dataframe["image_id"]
                == prediction["image_id"]
            )
            &
            (
                ground_truth_dataframe["class_name"]
                == prediction["predicted_class"]
            )
        ]

        best_iou = 0.0
        best_ground_truth_index = None

        prediction_box = (
            prediction["xmin"],
            prediction["ymin"],
            prediction["xmax"],
            prediction["ymax"]
        )

        for ground_truth_index, ground_truth in candidates.iterrows():
            if ground_truth_index in matched_ground_truth:
                continue

            ground_truth_box = (
                ground_truth["xmin"],
                ground_truth["ymin"],
                ground_truth["xmax"],
                ground_truth["ymax"]
            )

            current_iou = calculate_iou(
                prediction_box,
                ground_truth_box
            )

            if current_iou > best_iou:
                best_iou = current_iou
                best_ground_truth_index = ground_truth_index

        is_true_positive = (
            best_ground_truth_index is not None
            and best_iou >= iou_threshold
        )

        if is_true_positive:
            matched_ground_truth.add(
                best_ground_truth_index
            )

        result_records.append({
            "prediction_index": prediction_index,
            "image_id": prediction["image_id"],
            "predicted_class": prediction["predicted_class"],
            "confidence": prediction["final_confidence"],
            "best_iou": best_iou,
            "true_positive": int(is_true_positive),
            "false_positive": int(not is_true_positive)
        })

    true_positives = sum(
        record["true_positive"]
        for record in result_records
    )

    false_positives = sum(
        record["false_positive"]
        for record in result_records
    )

    false_negatives = (
        len(ground_truth_dataframe)
        - len(matched_ground_truth)
    )

    precision = (
        true_positives
        / max(true_positives + false_positives, 1)
    )

    recall = (
        true_positives
        / max(true_positives + false_negatives, 1)
    )

    f1_score = (
        2 * precision * recall
        / max(precision + recall, 1e-12)
    )

    return {
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "precision": precision,
        "recall": recall,
        "f1": f1_score,
        "matched_predictions": pd.DataFrame(result_records)
    }

In [ ]:
OPERATING_THRESHOLD = 0.20  # Replace with validation-selected value

external_detection_results = evaluate_pipeline_at_threshold(
    prediction_dataframe=pipeline_predictions_dataframe,
    ground_truth_dataframe=new_test_dataframe,
    confidence_threshold=OPERATING_THRESHOLD,
    iou_threshold=0.50
)

for metric in [
    "true_positives",
    "false_positives",
    "false_negatives",
    "precision",
    "recall",
    "f1"
]:
    value = external_detection_results[metric]

    if isinstance(value, float):
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value}")

In [ ]:
def calculate_ap(recall, precision):
    recall = np.concatenate(([0.0], recall, [1.0]))
    precision = np.concatenate(([0.0], precision, [0.0]))

    for index in range(
        len(precision) - 2,
        -1,
        -1
    ):
        precision[index] = max(
            precision[index],
            precision[index + 1]
        )

    changing_points = np.where(
        recall[1:] != recall[:-1]
    )[0]

    average_precision = np.sum(
        (recall[changing_points + 1]
         - recall[changing_points])
        * precision[changing_points + 1]
    )

    return average_precision

In [ ]:
def calculate_class_ap(
    prediction_dataframe,
    ground_truth_dataframe,
    class_name,
    iou_threshold=0.5
):
    class_ground_truth = ground_truth_dataframe[
        ground_truth_dataframe["class_name"] == class_name
    ]

    class_predictions = prediction_dataframe[
        prediction_dataframe["predicted_class"] == class_name
    ].sort_values(
        "final_confidence",
        ascending=False
    )

    number_of_ground_truth = len(class_ground_truth)

    if number_of_ground_truth == 0:
        return None

    matched_ground_truth = set()
    true_positives = []
    false_positives = []

    for _, prediction in class_predictions.iterrows():
        candidates = class_ground_truth[
            class_ground_truth["image_id"]
            == prediction["image_id"]
        ]

        prediction_box = (
            prediction["xmin"],
            prediction["ymin"],
            prediction["xmax"],
            prediction["ymax"]
        )

        best_iou = 0.0
        best_index = None

        for ground_truth_index, ground_truth in candidates.iterrows():
            if ground_truth_index in matched_ground_truth:
                continue

            ground_truth_box = (
                ground_truth["xmin"],
                ground_truth["ymin"],
                ground_truth["xmax"],
                ground_truth["ymax"]
            )

            current_iou = calculate_iou(
                prediction_box,
                ground_truth_box
            )

            if current_iou > best_iou:
                best_iou = current_iou
                best_index = ground_truth_index

        if best_index is not None and best_iou >= iou_threshold:
            matched_ground_truth.add(best_index)
            true_positives.append(1)
            false_positives.append(0)
        else:
            true_positives.append(0)
            false_positives.append(1)

    if len(class_predictions) == 0:
        return 0.0

    cumulative_tp = np.cumsum(true_positives)
    cumulative_fp = np.cumsum(false_positives)

    recall = cumulative_tp / number_of_ground_truth

    precision = cumulative_tp / np.maximum(
        cumulative_tp + cumulative_fp,
        1
    )

    return calculate_ap(recall, precision)

In [ ]:
present_external_classes = sorted(
    new_test_dataframe["class_name"].unique()
)

class_ap_results = {}

for class_name in present_external_classes:
    class_ap_results[class_name] = calculate_class_ap(
        pipeline_predictions_dataframe,
        new_test_dataframe,
        class_name,
        iou_threshold=0.50
    )

class_ap_dataframe = pd.DataFrame({
    "class_name": class_ap_results.keys(),
    "AP@0.5": class_ap_results.values()
}).sort_values(
    "AP@0.5",
    ascending=False
)

external_map50 = class_ap_dataframe["AP@0.5"].mean()

print(class_ap_dataframe.to_string(index=False))
print(f"\nExternal class-aware mAP@0.5: {external_map50:.4f}")

In [ ]:
matches = cghd_detection_results[
    "matched_predictions"
].copy()

gt_counts = (
    cghd_test_ground_truth
    .groupby("image_id")
    .size()
    .rename("ground_truths")
)

image_summary = (
    matches
    .groupby("image_id")
    .agg(
        true_positives=("true_positive", "sum"),
        false_positives=("false_positive", "sum")
    )
)

image_summary = (
    gt_counts
    .to_frame()
    .join(image_summary, how="left")
    .fillna(0)
)

image_summary["false_negatives"] = (
    image_summary["ground_truths"]
    - image_summary["true_positives"]
)

image_summary["f1"] = (
    2 * image_summary["true_positives"]
    / (
        2 * image_summary["true_positives"]
        + image_summary["false_positives"]
        + image_summary["false_negatives"]
    ).clip(lower=1)
)

best_examples = (
    image_summary[image_summary["ground_truths"] >= 5]
    .sort_values(
        ["f1", "ground_truths"],
        ascending=[False, False]
    )
    .head(2)
)

most_missed = (
    image_summary
    .sort_values(
        "false_negatives",
        ascending=False
    )
    .head(1)
)

most_false_positives = (
    image_summary
    .sort_values(
        "false_positives",
        ascending=False
    )
    .head(1)
)

selected_examples = pd.concat([
    best_examples,
    most_missed,
    most_false_positives
])

display(selected_examples)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import pandas as pd


# ------------------------------------------------------------
# Utility: optionally crop image borders
# ------------------------------------------------------------
def crop_image(image, left=0, top=0, right=0, bottom=0):
    w, h = image.size
    return image.crop((left, top, w - right, h - bottom))


# ------------------------------------------------------------
# Panel drawing
# ------------------------------------------------------------
def draw_cghd_panel_clean(
    axis,
    image_id,
    title_main,
    title_sub,
    prediction_dataframe,
    ground_truth_dataframe,
    confidence_threshold=0.20,
    iou_threshold=0.50,
    crop_margins=None
):
    image_ground_truth = ground_truth_dataframe[
        ground_truth_dataframe["image_id"] == image_id
    ]

    if len(image_ground_truth) == 0:
        axis.text(
            0.5, 0.5,
            f"Image not found:\n{image_id}",
            ha="center", va="center", fontsize=12
        )
        axis.axis("off")
        return

    image_path = Path(image_ground_truth.iloc[0]["original_image_path"])
    image = Image.open(image_path).convert("RGB")

    # Optional manual crop to remove surrounding desk/background
    if crop_margins is not None:
        image = crop_image(
            image,
            left=crop_margins.get("left", 0),
            top=crop_margins.get("top", 0),
            right=crop_margins.get("right", 0),
            bottom=crop_margins.get("bottom", 0),
        )

    axis.imshow(image)

    predictions, false_negatives = classify_predictions_for_image(
        prediction_dataframe=prediction_dataframe,
        ground_truth_dataframe=ground_truth_dataframe,
        image_id=image_id,
        confidence_threshold=confidence_threshold,
        iou_threshold=iou_threshold,
    )

    styles = {
        "true_positive": {
            "edgecolor": "green",
            "linewidth": 1.4,
        },
        "classification_error": {
            "edgecolor": "orange",
            "linewidth": 1.5,
        },
        "false_positive": {
            "edgecolor": "red",
            "linewidth": 1.4,
        },
    }

    for prediction in predictions:
        style = styles[prediction["status"]]

        xmin = prediction["xmin"]
        ymin = prediction["ymin"]
        width = prediction["xmax"] - xmin
        height = prediction["ymax"] - ymin

        rect = Rectangle(
            (xmin, ymin),
            width,
            height,
            fill=False,
            edgecolor=style["edgecolor"],
            linewidth=style["linewidth"],
        )
        axis.add_patch(rect)

        label = f'{prediction["predicted_class"]} {prediction["confidence"]:.2f}'
        if prediction["status"] == "classification_error":
            label += f'\nGT: {prediction["ground_truth_class"]}'

        axis.text(
            xmin,
            max(0, ymin - 3),
            label,
            fontsize=5.5,
            color=style["edgecolor"],
            bbox=dict(
                facecolor="white",
                alpha=0.75,
                pad=0.4,
                edgecolor="none"
            )
        )

    for _, ground_truth in false_negatives.iterrows():
        xmin = ground_truth["xmin"]
        ymin = ground_truth["ymin"]
        width = ground_truth["xmax"] - xmin
        height = ground_truth["ymax"] - ymin

        rect = Rectangle(
            (xmin, ymin),
            width,
            height,
            fill=False,
            edgecolor="red",
            linewidth=1.3,
            linestyle="--",
        )
        axis.add_patch(rect)

    axis.set_title(
        f"{title_main}\n{title_sub}",
        fontsize=11,
        pad=6
    )
    axis.axis("off")


# ------------------------------------------------------------
# Create improved 2x2 qualitative figure
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(13, 11), facecolor="white")
axes = axes.flatten()

# Panel (a): crop tighter because it has lots of background
draw_cghd_panel_clean(
    axis=axes[0],
    image_id="drafter_20_C233_D1_P3",
    title_main="(a) Perfect CGHD",
    title_sub="32 TP, 0 FP, 0 FN, F1 = 1.00",
    prediction_dataframe=cghd_pipeline_predictions,
    ground_truth_dataframe=cghd_test_ground_truth,
    crop_margins={"left": 40, "top": 80, "right": 50, "bottom": 40}
)

draw_cghd_panel_clean(
    axis=axes[1],
    image_id="drafter_18_C209_D2_P2",
    title_main="(b) Dense CGHD",
    title_sub="115 TP, 19 FP, 6 FN, F1 = 0.90",
    prediction_dataframe=cghd_pipeline_predictions,
    ground_truth_dataframe=cghd_test_ground_truth,
)

draw_cghd_panel_clean(
    axis=axes[2],
    image_id="drafter_18_C215_D1_P1",
    title_main="(c) Difficult CGHD",
    title_sub="5 TP, 10 FP, 32 FN, F1 = 0.19",
    prediction_dataframe=cghd_pipeline_predictions,
    ground_truth_dataframe=cghd_test_ground_truth,
)

# External panel
external_path = test_image_path
if external_path.exists():
    external_img = Image.open(external_path).convert("RGB")
    axes[3].imshow(external_img)
    axes[3].set_title(
        "(d) Digitize-HCD example\nUnsupported current sources predicted as voltage sources",
        fontsize=11,
        pad=6
    )
else:
    axes[3].text(
        0.5, 0.5,
        "External image not found",
        ha="center", va="center", fontsize=12
    )

axes[3].axis("off")

# Cleaner legend
fig.text(0.18, 0.03, "Green: correct prediction", color="green", fontsize=11, ha="center")
fig.text(0.42, 0.03, "Orange: incorrect class", color="orange", fontsize=11, ha="center")
fig.text(0.64, 0.03, "Red: false positive", color="red", fontsize=11, ha="center")
fig.text(0.84, 0.03, "Dashed red: missed component", color="red", fontsize=11, ha="center")

plt.subplots_adjust(
    left=0.03,
    right=0.97,
    top=0.96,
    bottom=0.07,
    wspace=0.08,
    hspace=0.12
)

output_path = Path("/content/gdrive/MyDrive/APS360/Project/qualitative_results_panel_clean.png")
plt.savefig(output_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved to:", output_path)